# Rust source → Spur graph coverage

Compare **physical Rust syntax** in `../spur-notebook` with its existing published graph. A separate parser supplies the denominator. Source hashes gate every coverage claim; stale and unindexed files remain visible.

Plan: freeze graph and source identities → independently inventory Rust items and calls → match exact source ranges → validate the matcher with calibration examples → chart coverage by item kind and source area, with file/symbol drilldowns.

This measures representation of source syntax. It does not measure executed tests, macro-expanded Rust, compiler type resolution, or panic proof completeness.

In [2]:
from pathlib import Path
import json, hashlib, subprocess, os, sys, shutil, time
from collections import Counter
import duckdb
ROOT = Path("/Users/kevintruong/project/spur-notebook")
OUT = Path("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-rust-graph-coverage")
OUT.mkdir(parents=True, exist_ok=True)
CURRENT = (ROOT / ".spur/graph/CURRENT").resolve(strict=True)
print("Python", sys.version.split()[0], "DuckDB", duckdb.__version__)
print("CURRENT", str(CURRENT))
print("snapshot entries", [(p.name, p.stat().st_size) for p in sorted(CURRENT.iterdir())] if CURRENT.is_dir() else CURRENT.stat().st_size)
old_manifest = json.loads(Path("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/source-manifest.json").read_text())
print("old source manifest type", type(old_manifest).__name__, "keys", list(old_manifest)[:8])
print("head", subprocess.check_output(["git","-C",str(ROOT),"rev-parse","HEAD"], text=True).strip())
print("cached syn crates", [p.name for p in Path("/Users/kevintruong/.cargo/registry/src").glob("*/syn-*")][:10])


Python 3.12.10 DuckDB 1.5.5
CURRENT /Users/kevintruong/project/spur-notebook/.spur/graph/cbc715ac26ad217292466f09775b8eff130b0d49885af4d6817035fecd40bbc1.parquet
snapshot entries [('code_symbols.parquet', 52922271), ('commits.parquet', 103704), ('diagnostics.parquet', 188101), ('edges.parquet', 8282606), ('edges_by_dst.parquet', 8598829), ('edges_unresolved.parquet', 2061523), ('file_manifests.parquet', 510294), ('files.parquet', 104765), ('manifest.json', 3221), ('nodes.parquet', 13487605), ('sections.parquet', 34875909), ('symbol_snapshots', 480), ('temporal_edges', 480), ('tombstones.parquet', 4225)]
old source manifest type list keys [{'path': '.cargo/config.toml', 'bytes': 32, 'sha256': 'b25e4a2d0a7562470f166e8082f1ff2ee01bcd01cc222935e4fcffb15febfc4f', 'git_blob_oid': 'c91c3f38b7b3ee09bdca168167855324db35608a', 'tracked': True, 'rust': False}, {'path': 'AGENTS.md', 'bytes': 4439, 'sha256': 'ef18d2d0274013c1ef3c6046efbd06b79f335096ef60a084609c6b14d7af0592', 'git_blob_oid': '0a72b9

In [3]:
GRAPH_MANIFEST = json.loads((CURRENT/"manifest.json").read_text())
print("graph manifest", json.dumps(GRAPH_MANIFEST, indent=2))
con = duckdb.connect()
for table in ["file_manifests", "nodes", "edges", "edges_unresolved", "diagnostics", "code_symbols"]:
    path = CURRENT/(table+".parquet")
    columns = con.execute("DESCRIBE SELECT * FROM read_parquet(?)", [str(path)]).fetchall()
    print("SCHEMA", table, columns)
print("Rust graph paths sample", con.execute("SELECT * FROM read_parquet(?) WHERE path LIKE '%.rs' LIMIT 2", [str(CURRENT/"file_manifests.parquet")]).fetchall())
print("symbol kinds", con.execute("SELECT symbol_kind,count(*) FROM read_parquet(?) GROUP BY 1 ORDER BY 2 DESC", [str(CURRENT/"nodes.parquet")]).fetchall())
print("Rust symbols sample", con.execute("SELECT * FROM read_parquet(?) WHERE file_path LIKE '%.rs' LIMIT 6", [str(CURRENT/"nodes.parquet")]).fetchall())
print("rust crate dependencies can be loaded in notebook; will use syn 2 full AST without macro expansion.")


graph manifest {
  "graph_index_version": "4",
  "schema_version": "spur-graph-schema-v10",
  "manifest_version": "c34989d757962afdb5a61b8f8804a08f4f13008b637e34297c1369ebf9a87d7e",
  "graph_content_hash": "cbc715ac26ad217292466f09775b8eff130b0d49885af4d6817035fecd40bbc1",
  "indexed_commit_oid": null,
  "extractor_version": "2026-08-30-html-css-extraction-v1",
  "complete": true,
  "row_counts": {
    "nodes": 273782,
    "edges": 538095,
    "edges_by_dst": 538095,
    "edges_unresolved": 155197,
    "files": 4255,
    "file_manifests": 4255,
    "tombstones": 263,
    "commits": 1378,
    "symbol_snapshots": 705394,
    "temporal_edges": 727828,
    "diagnostics": 10039
  },
  "sidecar_complete": true,
  "sidecar_row_counts": {
    "section_bodies": 15151,
    "code_symbols": 254560
  },
  "parquet_writer": {
    "compression": "zstd-3",
    "row_group_size": 16384
  },
  "edges_by_dst_present": true,
  "temporal_shards": [
    {
      "shard_idx": 0,
      "commit_time_min": 177946

SCHEMA file_manifests [('stable_file_id', 'VARCHAR', 'YES', None, None, None), ('path', 'VARCHAR', 'YES', None, None, None), ('content_oid', 'VARCHAR', 'YES', None, None, None), ('node_ids', 'BIGINT[]', 'YES', None, None, None)]
SCHEMA nodes [('stable_symbol_id', 'VARCHAR', 'YES', None, None, None), ('node_id', 'BIGINT', 'YES', None, None, None), ('file_path', 'VARCHAR', 'YES', None, None, None), ('byte_range_start', 'BIGINT', 'YES', None, None, None), ('byte_range_end', 'BIGINT', 'YES', None, None, None), ('line_start', 'INTEGER', 'YES', None, None, None), ('line_end', 'INTEGER', 'YES', None, None, None), ('entity_name', 'VARCHAR', 'YES', None, None, None), ('qualified_name', 'VARCHAR', 'YES', None, None, None), ('symbol_kind', 'VARCHAR', 'YES', None, None, None), ('anchor_hash', 'VARCHAR', 'YES', None, None, None), ('enclosing_scope', 'VARCHAR', 'YES', None, None, None)]
SCHEMA edges [('source_stable_id', 'VARCHAR', 'YES', None, None, None), ('target_stable_id', 'VARCHAR', 'YES', Non

In [5]:
def dump(name, obj):
    (OUT/name).write_text(json.dumps(obj, indent=2, ensure_ascii=False)+"\n")
def sha(data): return hashlib.sha256(data).hexdigest()
def git(*args):
    return subprocess.check_output(["git","-C",str(ROOT),*args])
state_before = {"head":git("rev-parse","HEAD").decode().strip(), "status":git("status","--porcelain=v1","-z").decode(), "graph_current":str(CURRENT), "graph_manifest_sha256":sha((CURRENT/"manifest.json").read_bytes())}
dump("project-state-before.json",state_before)
paths = sorted(set(git("ls-files","--cached","--others","--exclude-standard","-z","--","*.rs").decode().split("\0"))-{""})
source_manifest=[]
snapshot=OUT/"source-snapshot"
for path in paths:
    p=ROOT/path
    if not p.is_file(): continue
    b=p.read_bytes()
    d=snapshot/path
    d.parent.mkdir(parents=True,exist_ok=True); d.write_bytes(b)
    source_manifest.append({"path":path,"bytes":len(b),"lines":len(b.splitlines()),"sha256":sha(b),"git_blob_oid":hashlib.sha1(b"blob "+str(len(b)).encode()+b"\0"+b).hexdigest()})
dump("source-manifest.json",source_manifest)
dump("graph-manifest.json",GRAPH_MANIFEST)
for table in ["file_manifests","nodes"]:
    target=OUT/("graph-"+table+".parquet")
    col="path" if table=="file_manifests" else "file_path"
    con.read_parquet(str(CURRENT/(table+".parquet"))).filter(col+" LIKE '%.rs'").write_parquet(str(target))
con.execute("CREATE OR REPLACE VIEW graph_nodes AS SELECT * FROM read_parquet('"+str(OUT/"graph-nodes.parquet")+"')")
for table in ["edges","edges_unresolved"]:
    con.sql("SELECT e.* FROM read_parquet(\'"+str(CURRENT/(table+".parquet"))+"\') e SEMI JOIN graph_nodes n ON n.stable_symbol_id=e.source_stable_id").write_parquet(str(OUT/("graph-"+table+".parquet")))
fm = {r[0]:r[1] for r in con.execute("SELECT path,content_oid FROM read_parquet(?)",[str(OUT/"graph-file_manifests.parquet")]).fetchall()}
file_states=Counter("unindexed" if r["path"] not in fm else "source_match" if fm[r["path"]]==r["git_blob_oid"] else "stale" for r in source_manifest)
dump("snapshot-summary.json",{"rust_files":len(source_manifest),"bytes":sum(r["bytes"] for r in source_manifest),"lines":sum(r["lines"] for r in source_manifest),"file_states":dict(file_states),"graph_rust_files":len(fm),"graph_only_paths":sorted(set(fm)-{r["path"] for r in source_manifest})})
print("Current Rust inventory",len(source_manifest),"files",sum(r["lines"] for r in source_manifest),"lines")
print("Published graph source freshness",dict(file_states),"graph-only paths",len(set(fm)-{r["path"] for r in source_manifest}))
print("Rust graph kinds",con.execute("SELECT symbol_kind,count(*) FROM graph_nodes GROUP BY 1 ORDER BY 2 DESC").fetchall())


Current Rust inventory 689 files 366382 lines
Published graph source freshness {'source_match': 651, 'unindexed': 21, 'stale': 17} graph-only paths 0
Rust graph kinds [('function', 9097), ('field', 6379), ('method', 3220), ('enum_variant', 1849), ('struct', 1529), ('impl', 952), ('constant', 889), ('module', 838), ('enum', 401), ('type_alias', 98), ('trait', 35), ('macro', 3)]


In [3]:
:dep syn = { version = "=2.0.118", features = ["full", "visit", "extra-traits"] }
:dep proc-macro2 = { version = "1", features = ["span-locations"] }
:dep serde_json = "1"
println!("Independent Rust parser dependencies loaded: syn 2.0.118, proc-macro2 span-locations, serde_json");

The type of the variable spur was redefined, so was lost.


Independent Rust parser dependencies loaded: syn 2.0.118, proc-macro2 span-locations, serde_json


In [7]:
use syn::spanned::Spanned;
use syn::visit::{self, Visit};
use proc_macro2::Span;
use serde_json::{json, Value};

struct RustInventory {
    file: String, rows: Vec<Value>, test_context: bool, cfg_context: bool,
}
impl RustInventory {
    fn attr_flags(attrs: &[syn::Attribute]) -> (bool, bool) {
        let test = attrs.iter().any(|a| a.path().segments.last().map(|s|s.ident == "test").unwrap_or(false)) ||
            attrs.iter().any(|a| a.path().is_ident("cfg") && a.parse_args::<syn::Path>().map(|p|p.is_ident("test")).unwrap_or(false));
        let cfg = attrs.iter().any(|a| a.path().is_ident("cfg") || a.path().is_ident("cfg_attr"));
        (test, cfg)
    }
    fn push(&mut self, row_type: &str, kind: &str, name: String,
            span: Span, ident: Option<Span>, body: Option<bool>) {
        let range = span.byte_range();
        let idrange = ident.map(|s| s.byte_range());
        self.rows.push(json!({
            "file_path": self.file, "row_type":row_type, "kind":kind, "name":name,
            "start":range.start, "core_start":range.start, "end":range.end,
            "ident_start":idrange.as_ref().map(|r|r.start),
            "ident_end":idrange.as_ref().map(|r|r.end),
            "line_start":span.start().line, "line_end":span.end().line,
            "test_context":self.test_context, "cfg_context":self.cfg_context,
            "has_body":body,
        }));
    }
    fn named(&mut self, kind:&str, id:&syn::Ident, span:Span, body:Option<bool>) {
        self.push("symbol",kind,id.to_string(),span,Some(id.span()),body);
    }
}
impl<'ast> Visit<'ast> for RustInventory {
    fn visit_item(&mut self, item:&'ast syn::Item) {
        let attrs: &[syn::Attribute] = match item {
            syn::Item::Const(x)=>&x.attrs, syn::Item::Enum(x)=>&x.attrs,
            syn::Item::ExternCrate(x)=>&x.attrs, syn::Item::Fn(x)=>&x.attrs,
            syn::Item::ForeignMod(x)=>&x.attrs, syn::Item::Impl(x)=>&x.attrs,
            syn::Item::Macro(x)=>&x.attrs, syn::Item::Mod(x)=>&x.attrs,
            syn::Item::Static(x)=>&x.attrs, syn::Item::Struct(x)=>&x.attrs,
            syn::Item::Trait(x)=>&x.attrs, syn::Item::TraitAlias(x)=>&x.attrs,
            syn::Item::Type(x)=>&x.attrs, syn::Item::Union(x)=>&x.attrs,
            syn::Item::Use(x)=>&x.attrs, _=>&[],
        };
        let prev=(self.test_context,self.cfg_context);
        let flags=Self::attr_flags(attrs);
        self.test_context |= flags.0; self.cfg_context |= flags.1;
        match item {
            syn::Item::Fn(x)=>self.named("function",&x.sig.ident,x.span(),Some(true)),
            syn::Item::Struct(x)=>self.named("struct",&x.ident,x.span(),None),
            syn::Item::Enum(x)=>self.named("enum",&x.ident,x.span(),None),
            syn::Item::Union(x)=>self.named("union",&x.ident,x.span(),None),
            syn::Item::Trait(x)=>self.named("trait",&x.ident,x.span(),None),
            syn::Item::TraitAlias(x)=>self.named("trait_alias",&x.ident,x.span(),None),
            syn::Item::Type(x)=>self.named("type_alias",&x.ident,x.span(),None),
            syn::Item::Const(x)=>self.named("constant",&x.ident,x.span(),None),
            syn::Item::Static(x)=>self.named("static",&x.ident,x.span(),None),
            syn::Item::Mod(x)=>self.named("module",&x.ident,x.span(),None),
            syn::Item::Macro(x) if x.ident.is_some()=>self.named("macro_definition",x.ident.as_ref().unwrap(),x.span(),None),
            syn::Item::Impl(x)=>self.push("symbol","impl","<impl>".into(),x.span(),None,None),
            syn::Item::ForeignMod(x)=>self.push("symbol","extern_block","<extern>".into(),x.span(),None,None),
            _=>(),
        }
        visit::visit_item(self,item);
        (self.test_context,self.cfg_context)=prev;
    }
    fn visit_impl_item(&mut self, item:&'ast syn::ImplItem) {
        let prev=(self.test_context,self.cfg_context);
        let attrs=match item {syn::ImplItem::Fn(x)=>&x.attrs, syn::ImplItem::Const(x)=>&x.attrs, syn::ImplItem::Type(x)=>&x.attrs, _=>&[][..]};
        let flags=Self::attr_flags(attrs);self.test_context|=flags.0;self.cfg_context|=flags.1;
        match item {
            syn::ImplItem::Fn(x)=>self.named("method",&x.sig.ident,x.span(),Some(true)),
            syn::ImplItem::Const(x)=>self.named("associated_constant",&x.ident,x.span(),None),
            syn::ImplItem::Type(x)=>self.named("associated_type",&x.ident,x.span(),None),
            _=>(),
        }
        visit::visit_impl_item(self,item);
        (self.test_context,self.cfg_context)=prev;
    }
    fn visit_trait_item(&mut self, item:&'ast syn::TraitItem) {
        let prev=(self.test_context,self.cfg_context);
        let attrs=match item {syn::TraitItem::Fn(x)=>&x.attrs, syn::TraitItem::Const(x)=>&x.attrs, syn::TraitItem::Type(x)=>&x.attrs, _=>&[][..]};
        let flags=Self::attr_flags(attrs);self.test_context|=flags.0;self.cfg_context|=flags.1;
        match item {
            syn::TraitItem::Fn(x)=>self.named("trait_method",&x.sig.ident,x.span(),Some(x.default.is_some())),
            syn::TraitItem::Const(x)=>self.named("trait_constant",&x.ident,x.span(),None),
            syn::TraitItem::Type(x)=>self.named("trait_type",&x.ident,x.span(),None),
            _=>(),
        }
        visit::visit_trait_item(self,item);
        (self.test_context,self.cfg_context)=prev;
    }
    fn visit_foreign_item(&mut self, item:&'ast syn::ForeignItem) {
        match item {
            syn::ForeignItem::Fn(x)=>self.named("foreign_function",&x.sig.ident,x.span(),Some(false)),
            syn::ForeignItem::Static(x)=>self.named("foreign_static",&x.ident,x.span(),None),
            syn::ForeignItem::Type(x)=>self.named("foreign_type",&x.ident,x.span(),None),
            _=>(),
        }
        visit::visit_foreign_item(self,item);
    }
    fn visit_variant(&mut self,x:&'ast syn::Variant) {
        self.named("enum_variant",&x.ident,x.span(),None);
        visit::visit_variant(self,x);
    }
    fn visit_field(&mut self,x:&'ast syn::Field) {
        if let Some(id)=&x.ident {self.named("field",id,x.span(),None);}
        else {self.push("symbol","tuple_field","<unnamed>".into(),x.span(),None,None);}
        visit::visit_field(self,x);
    }
    fn visit_expr(&mut self,x:&'ast syn::Expr) {
        match x {
            syn::Expr::Call(c)=>{
                let name = match c.func.as_ref() {
                    syn::Expr::Path(p)=>p.path.segments.last().map(|s|s.ident.to_string()).unwrap_or_default(),
                    _=>"<expression>".into(),
                };
                self.push("callsite","call",name,c.span(),None,None);
            },
            syn::Expr::MethodCall(c)=>{
                self.push("callsite","method_call",c.method.to_string(),c.span(),Some(c.method.span()),None);
                self.rows.last_mut().unwrap()["core_start"]=json!(c.receiver.span().byte_range().start);
            },
            syn::Expr::Index(c)=>self.push("operation","index","[]".into(),c.span(),None,None),
            syn::Expr::Closure(c)=>self.push("body","closure","<closure>".into(),c.span(),None,Some(true)),
            syn::Expr::Async(c)=>self.push("body","async_block","<async>".into(),c.span(),None,Some(true)),
            syn::Expr::Const(c)=>self.push("body","const_block","<const>".into(),c.span(),None,Some(true)),
            _=>(),
        }
        visit::visit_expr(self,x);
    }
    fn visit_macro(&mut self,x:&'ast syn::Macro) {
        if !x.path.is_ident("macro_rules") {
            let name=x.path.segments.iter().map(|s|s.ident.to_string()).collect::<Vec<_>>().join("::");
            self.push("callsite","macro_invocation",name,x.span(),None,None);
        }
        // syn intentionally treats macro argument token trees as opaque.
        visit::visit_macro(self,x);
    }
}
fn inventory_rust(file:&str, source:&str)->Result<Vec<Value>,syn::Error>{
    let ast=syn::parse_file(source)?;
    let mut v=RustInventory{file:file.into(),rows:Vec::new(),test_context:false,cfg_context:false};
    v.visit_file(&ast); Ok(v.rows)
}
let calibration = "// é 🦀\n#[cfg(test)]\npub fn r#loop() { fn nested() {} nested(); let é = 1; let f=||é; }\nstruct S { field: u8 }\nenum E { A(u8), B { x: bool } }\ntrait T { fn required(&self); fn default(&self) {} }\nimpl S { fn method(&self) {} }\nmacro_rules! opaque { () => { fn generated() {} } }\nfn probes() { let s = \"é\"; let _ = s.trim().len(); #[cfg(test)] s.trim(); assert!(true); }\nunsafe extern \"C\" { fn foreign_probe(); }\n";
let cal=inventory_rust("calibration.rs",calibration)?;
assert_eq!(cal.iter().filter(|r|r["kind"]=="function").count(),3);
assert_eq!(cal.iter().filter(|r|r["kind"]=="trait_method").count(),2);
assert_eq!(cal.iter().filter(|r|r["kind"]=="macro_definition").count(),1);
assert!(!cal.iter().any(|r|r["name"]=="generated"));
for r in cal.iter().filter(|r| !r["ident_start"].is_null()) {
    let start=r["ident_start"].as_u64().unwrap() as usize;
    let end=r["ident_end"].as_u64().unwrap() as usize;
    assert_eq!(&calibration[start..end],r["name"].as_str().unwrap());
}
let coverage_out=std::path::Path::new("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-rust-graph-coverage");
std::fs::write(coverage_out.join("calibration.rs"),calibration)?;
std::fs::write(coverage_out.join("calibration-ast.json"),serde_json::to_string_pretty(&cal)?)?;
let manifest:Value=serde_json::from_str(&std::fs::read_to_string(coverage_out.join("source-manifest.json"))?)?;
let mut all=Vec::new(); let mut parses=Vec::new();
for file in manifest.as_array().unwrap() {
    let path=file["path"].as_str().unwrap();
    let text=std::fs::read_to_string(coverage_out.join("source-snapshot").join(path))?;
    match inventory_rust(path,&text) {
        Ok(rows)=>{parses.push(json!({"file_path":path,"status":"parsed","rows":rows.len()}));all.extend(rows);}
        Err(e)=>parses.push(json!({"file_path":path,"status":"parse_error","error":e.to_string()})),
    }
}
std::fs::write(coverage_out.join("rust-ast.json"),serde_json::to_string(&all)?)?;
std::fs::write(coverage_out.join("rust-parse-results.json"),serde_json::to_string_pretty(&parses)?)?;
println!("Independent physical AST rows: {}; parsed files: {}; parse errors: {}. Calibration passed (Unicode, raw identifiers, nested functions, trait declarations, opaque macros).",
    all.len(),parses.iter().filter(|r|r["status"]=="parsed").count(),parses.iter().filter(|r|r["status"]=="parse_error").count());


Independent physical AST rows: 205726; parsed files: 688; parse errors: 1. Calibration passed (Unicode, raw identifiers, nested functions, trait declarations, opaque macros).


In [6]:
import pandas as pd
ast = pd.read_json(OUT/"rust-ast.json")
parses = pd.read_json(OUT/"rust-parse-results.json")
nodes = con.read_parquet(str(OUT/"graph-nodes.parquet")).df()
print("AST counts",ast.groupby(["row_type","kind"]).size().to_string())
print("Parse errors",parses[parses.status!="parsed"].to_dict("records"))
print("Rust modules sample",nodes[nodes.symbol_kind=="module"].head(6).to_dict("records"))
print("Trait methods sample",nodes[(nodes.file_path.str.endswith("compiler_evidence.rs")) & (nodes.symbol_kind=="method")].head(2).to_dict("records"))
print("AST callable sample",ast[ast.kind.isin(["function","method","trait_method"])].head(4).to_dict("records"))
print("stale/unindexed sample",[{"path":r["path"],"status":"unindexed" if r["path"] not in fm else "stale"} for r in source_manifest if fm.get(r["path"])!=r["git_blob_oid"]][:12])


AST counts row_type   kind               
body       async_block              461
           closure                 8346
           const_block                1
callsite   call                   55381
           macro_invocation       28362
           method_call            81946
operation  index                   4722
symbol     associated_constant       14
           associated_type           32
           constant                 811
           enum                     421
           enum_variant            1932
           extern_block               3
           field                   6537
           foreign_function           9
           function                9306
           impl                    1006
           macro_definition           3
           method                  3291
           module                   864
           static                    56
           struct                  1584
           trait                     40
           trait_method              8

In [7]:
from collections import defaultdict
ast_rows = json.loads((OUT/"rust-ast.json").read_text())
source_by_path={r["path"]:r for r in source_manifest}
file_state={p:("unindexed" if p not in fm else "source_match" if fm[p]==r["git_blob_oid"] else "stale") for p,r in source_by_path.items()}
kind_map = {"trait_method":"method","associated_constant":"constant","trait_constant":"constant","associated_type":"type_alias","trait_type":"type_alias","static":"constant","foreign_static":"constant","foreign_function":"function","foreign_type":"type_alias","macro_definition":"macro"}
by_key=defaultdict(list)
for n in nodes.to_dict("records"):
    by_key[(n["file_path"],n["symbol_kind"],n["entity_name"].removeprefix("r#"))].append(n)
symbol_results=[]
for r in ast_rows:
    if r["row_type"]!="symbol":continue
    result=dict(r)
    result["file_status"]=file_state[r["file_path"]]
    graph_kind=kind_map.get(r["kind"],r["kind"])
    candidates=by_key.get((r["file_path"],graph_kind,r["name"].removeprefix("r#")),[])
    if r["ident_start"] is not None:
        candidates=[n for n in candidates if r["start"]<=n["byte_range_start"]<=r["ident_start"] and n["byte_range_start"]<=r["ident_start"]<r["ident_end"]<=n["byte_range_end"]==r["end"]]
    elif r["kind"]=="impl":
        candidates=[n for n in nodes.to_dict("records") if n["file_path"]==r["file_path"] and n["symbol_kind"]=="impl" and n["byte_range_start"]>=r["start"] and n["byte_range_end"]==r["end"]]
    else: candidates=[]
    result["candidate_count"]=len(candidates)
    result["matched_symbol_id"]=candidates[0]["stable_symbol_id"] if len(candidates)==1 else None
    result["coverage_status"]=result["file_status"] if result["file_status"]!="source_match" else "represented" if len(candidates)==1 else "ambiguous" if candidates else "no_matching_symbol"
    symbol_results.append(result)
symbols=pd.DataFrame(symbol_results)
print("symbol coverage",symbols.groupby(["kind","coverage_status"]).size().unstack(fill_value=0).to_string())
print("No matching callables examples",symbols[(symbols.coverage_status=="no_matching_symbol") & symbols.kind.isin(["function","method","trait_method"])][["file_path","kind","name","start","end","ident_start"]].head(20).to_string(index=False))
dump("symbol-coverage.json",symbol_results)


symbol coverage coverage_status      no_matching_symbol  represented  stale  unindexed
kind                                                                  
associated_constant                   0           14      0          0
associated_type                       0           32      0          0
constant                              0          776     25         10
enum                                  0          371     30         20
enum_variant                          0         1590    259         83
extern_block                          3            0      0          0
field                                 0         5718    662        157
foreign_function                      9            0      0          0
function                              0         8633    471        202
impl                                  0          893     61         52
macro_definition                      0            3      0          0
method                                0         2826    309  

In [8]:
# Inspect gaps and the existing experiment's public request shape.
fresh = symbols[symbols.file_status=="source_match"]
print("Fresh syntax symbols:",len(fresh),"represented:",(fresh.coverage_status=="represented").sum())
print("Remaining syntax classes:",fresh[fresh.coverage_status!="represented"].groupby("kind").size().to_dict())
for r in fresh[fresh.kind.isin(["tuple_field","foreign_function"])].head(4).to_dict("records"):
    b=(snapshot/r["file_path"]).read_bytes()
    nearby=nodes[(nodes.file_path==r["file_path"]) & (nodes.byte_range_start>=r["start"]-60) & (nodes.byte_range_start<=r["end"]+20)]
    print("GAP",r["file_path"],r["kind"],r["start"],r["end"],b[r["start"]:r["end"]].decode(),nearby[["entity_name","symbol_kind","byte_range_start","byte_range_end"]].to_dict("records"))
old=Path("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation")
print("Known per-file request artifacts",[(p.name,[q.name for q in p.iterdir()][:8]) for p in (old/"per-file-private").iterdir() if p.is_dir()][:5])


Fresh syntax symbols: 23573 represented: 23197
Remaining syntax classes: {'extern_block': 3, 'foreign_function': 9, 'tuple_field': 364}
GAP docs/reviews/2026-09-29-catalog-detach-fix/catalog-final-revision/catalog_tokens.before.rs tuple_field 330 336 String [{'entity_name': 'Identifier', 'symbol_kind': 'enum_variant', 'byte_range_start': 319, 'byte_range_end': 337}, {'entity_name': 'Token', 'symbol_kind': 'enum', 'byte_range_start': 291, 'byte_range_end': 400}, {'entity_name': 'Keyword', 'symbol_kind': 'enum_variant', 'byte_range_start': 343, 'byte_range_end': 358}]
GAP docs/reviews/2026-09-29-catalog-detach-fix/catalog-final-revision/catalog_tokens.before.rs tuple_field 351 357 String [{'entity_name': 'Literal', 'symbol_kind': 'enum_variant', 'byte_range_start': 364, 'byte_range_end': 379}, {'entity_name': 'Identifier', 'symbol_kind': 'enum_variant', 'byte_range_start': 319, 'byte_range_end': 337}, {'entity_name': 'Token', 'symbol_kind': 'enum', 'byte_range_start': 291, 'byte_range_en

In [9]:
print("snapshot request",json.loads((old/"per-file-private/jute-state/snapshot.request.json").read_text()))
print("snapshot command",json.loads((old/"per-file-private/jute-state/snapshot.command.json").read_text()))


snapshot request {'command': 'snapshot', 'root': '/private/tmp/spur-notebook-panic-evaluation-20261102/jute-state', 'source_paths': ['jute-notebook/src-tauri/src/state.rs']}
snapshot command {'binary_sha256': '984ee7bc02555b9c831addac41ce44d0bf323bf685f011c52078f815363eb5cd', 'input_sha256': 'b96dada3ae57bd2dcd9adfcdea9e61b94a8c323050cb322eb66a1410f06e9998', 'exit': 0, 'elapsed_seconds': 1.723369836807251, 'environment': {'RUSTUP_TOOLCHAIN': '1.94.1-aarch64-apple-darwin', 'TMPDIR': '/private/tmp'}}


In [10]:
import concurrent.futures, gzip
BIN = Path("/private/tmp/r8g-bd-3d9-b2ae97ed-target/debug/examples/panic_evidence")
BIN_SHA = sha(BIN.read_bytes())
assert BIN_SHA=="984ee7bc02555b9c831addac41ce44d0bf323bf685f011c52078f815363eb5cd"
PRIVATE=Path("/private/tmp/rust-graph-coverage-bd-33a")
PRIVATE.mkdir(exist_ok=True)
(OUT/"current-extractor").mkdir(exist_ok=True)
def extract_one(item):
    path=item["path"]
    token=hashlib.sha256(path.encode()).hexdigest()[:20]
    root=PRIVATE/token
    p=root/path
    p.parent.mkdir(parents=True,exist_ok=True)
    source=(OUT/"calibration.rs").read_bytes() if path=="calibration.rs" else (snapshot/path).read_bytes()
    p.write_bytes(source)
    request={"command":"snapshot","root":str(root),"source_paths":[path]}
    inp=json.dumps(request).encode()
    started=time.monotonic()
    env=os.environ.copy();env.update(RUSTUP_TOOLCHAIN="1.94.1-aarch64-apple-darwin",TMPDIR="/private/tmp")
    record={"path":path,"input_sha256":sha(inp),"source_sha256":sha(source),"request":request,"binary_sha256":BIN_SHA}
    try:
        run=subprocess.run([str(BIN)],input=inp,stdout=subprocess.PIPE,stderr=subprocess.PIPE,cwd=PRIVATE,env=env,timeout=30)
        record.update(exit=run.returncode,elapsed_seconds=time.monotonic()-started,stdout_sha256=sha(run.stdout),stderr=run.stderr.decode(errors="replace"))
        if run.returncode==0:
            obj=json.loads(run.stdout)
            assert set(f["file_path"] for f in obj["graph"]["files"])=={path}, "unexpected extraction input"
            (OUT/"current-extractor"/(token+".json.gz")).write_bytes(gzip.compress(run.stdout,mtime=0))
            record.update(status="success",artifact=token+".json.gz",graph_hash=obj["graph"]["graph_content_hash"],symbols=len(obj["graph"]["symbols"]),methods=len(obj["method_sites"]["sites"]),macros=len(obj["graph"]["macro_invocations"]["rows"]))
        else:record["status"]="failed"
    except subprocess.TimeoutExpired as e:
        record.update(status="timeout",elapsed_seconds=time.monotonic()-started)
    except Exception as e:
        record.update(status="error",error=str(e),elapsed_seconds=time.monotonic()-started)
    return record
calibration_record=extract_one({"path":"calibration.rs"})
assert calibration_record["status"]=="success",calibration_record
dump("calibration-current-extractor.json",calibration_record)
pool=concurrent.futures.ThreadPoolExecutor(max_workers=6)
extraction_started=time.monotonic()
extraction_futures=[pool.submit(extract_one,item) for item in source_manifest]
dump("current-extractor-provenance.json",{"binary":str(BIN),"sha256":BIN_SHA,"source_commit":"848a29f39511cbd27091beb6c171b97e179f353a","mode":"file-local extraction, no project-wide target resolution","concurrency":6,"timeout_seconds_per_file":30,"files":len(source_manifest)})
print("Calibration extraction succeeded in",round(calibration_record["elapsed_seconds"],2),"seconds.")
print("Queued",len(extraction_futures),"file-local extractions against captured bytes. Published graph CURRENT unchanged.")


Calibration extraction succeeded in 0.1 seconds.
Queued 689 file-local extractions against captured bytes. Published graph CURRENT unchanged.


In [11]:
done=[f.result() for f in extraction_futures if f.done()]
print("Current extraction progress",len(done),"/",len(extraction_futures),Counter(r["status"] for r in done),"elapsed",round(time.monotonic()-extraction_started,1),"seconds")
if len(done)==len(extraction_futures):
    extraction_records=done
    dump("current-extractor-results.json",extraction_records)
    pool.shutdown(wait=False)
print("Non-success", [r for r in done if r["status"]!="success"][:3])


Current extraction progress 213 / 689 Counter({'success': 211, 'timeout': 2}) elapsed 50.4 seconds
Non-success [{'path': 'jute-notebook/src-tauri/src/commands.rs', 'input_sha256': 'a1492ea3cc6e11cb08408ae5df242e015feab63c6ba5005a30f88e3a9cfb2946', 'source_sha256': 'f003d9530734ca7ad62996eded53a16854b500e49e8537393da95faa78ed569b', 'request': {'command': 'snapshot', 'root': '/private/tmp/rust-graph-coverage-bd-33a/89eca9177f502bf3dd16', 'source_paths': ['jute-notebook/src-tauri/src/commands.rs']}, 'binary_sha256': '984ee7bc02555b9c831addac41ce44d0bf323bf685f011c52078f815363eb5cd', 'status': 'timeout', 'elapsed_seconds': 30.00618866601144}, {'path': 'jute-notebook/src-tauri/src/notebook_store.rs', 'input_sha256': 'f17b0873c0981d1715022d2123d054122e6af734d5360645a182709b0a4b073b', 'source_sha256': '3808f704403c1f2d9216a341b4cbb66362989207712905cdd42818d250c1ab07', 'request': {'command': 'snapshot', 'root': '/private/tmp/rust-graph-coverage-bd-33a/ec11f7bf74cd0e4e8999', 'source_paths': ['j

In [12]:
cal_graph=json.loads(gzip.decompress((OUT/"current-extractor"/calibration_record["artifact"]).read_bytes()))
print("Current graph symbol",cal_graph["graph"]["symbols"][:2])
print("Current method rows",cal_graph["method_sites"]["sites"][:1])
print("Current macro rows",cal_graph["graph"]["macro_invocations"]["rows"][:2])
progress=[f.result() for f in extraction_futures if f.done()]
print("Current extraction progress",len(progress),"/",len(extraction_futures),dict(Counter(r["status"] for r in progress)))


Current graph symbol [{'stable_symbol_id': '604d61f34612805b', 'file_path': 'calibration.rs', 'byte_range': [24, 92], 'line_range': [3, 3], 'entity_name': 'r#loop', 'qualified_name': 'r#loop', 'symbol_kind': 'function', 'anchor_hash': '14047764701812327394', 'enclosing_scope': None}, {'stable_symbol_id': '9e7fe49a17a2d45a', 'file_path': 'calibration.rs', 'byte_range': [42, 56], 'line_range': [3, 3], 'entity_name': 'nested', 'qualified_name': 'r#loop::nested', 'symbol_kind': 'function', 'anchor_hash': '3300995882221685821', 'enclosing_scope': 'r#loop'}]
Current method rows []
Current macro rows []
Current extraction progress 448 / 689 {'success': 444, 'timeout': 4}


In [18]:
calibration_record=extract_one({"path":"calibration.rs"})
dump("calibration-current-extractor.json",calibration_record)
cal_graph=json.loads(gzip.decompress((OUT/"current-extractor"/calibration_record["artifact"]).read_bytes()))
print("Calibration methods",cal_graph["method_sites"]["sites"])
print("Calibration macros",cal_graph["graph"]["macro_invocations"]["rows"])
progress=[f.result() for f in extraction_futures if f.done()]
print("Progress",len(progress),len(extraction_futures),dict(Counter(r["status"] for r in progress)))


Calibration methods [{'site_id': 'db144166f52a00354a00a90804d26cc8077b887a488d081c853a31c2f1d9580b', 'file_path': 'calibration.rs', 'source_oid': 'cfa62d6f1eb037afddf280abcd3a3a0a676528b1', 'byte_range': [320, 328], 'method_byte_range': [322, 326], 'receiver_byte_range': [320, 321], 'line_range': [9, 9], 'call_text': 's.trim()', 'method_name': 'trim', 'receiver_text': 's', 'caller_stable_symbol_id': '034b1dbb66308b7b', 'lexical_phase': 'function_body_syntax', 'lexical_owners': [{'syntax_kind': 'function_item', 'byte_range': [284, 375], 'line_range': [9, 9]}, {'syntax_kind': 'source_file', 'byte_range': [0, 418], 'line_range': [1, 10]}], 'compiler_target': 'unknown'}, {'site_id': '3fe9fbb8492cd3a2e03e8ee190e622d95d42fafad523418c97fdcf40b15effcb', 'file_path': 'calibration.rs', 'source_oid': 'cfa62d6f1eb037afddf280abcd3a3a0a676528b1', 'byte_range': [320, 334], 'method_byte_range': [329, 332], 'receiver_byte_range': [320, 328], 'line_range': [9, 9], 'call_text': 's.trim().len()', 'method_

In [14]:
# Complete captures are required before comparing current-extractor coverage.
done=[f.result() for f in extraction_futures if f.done()]
print("Extraction",len(done),"/",len(extraction_futures),dict(Counter(r["status"] for r in done)))
assert len(done)==len(extraction_futures), "extractions still running; rerun this cell when all have finished"
extraction_records=done
dump("current-extractor-results.json",extraction_records)
pool.shutdown(wait=False)
print("Timeout paths",[r["path"] for r in done if r["status"]=="timeout"])
print("Total capture bytes",sum(p.stat().st_size for p in (OUT/"current-extractor").glob("*.gz")))


Extraction 689 / 689 {'success': 683, 'timeout': 6}
Timeout paths ['jute-notebook/src-tauri/src/commands.rs', 'jute-notebook/src-tauri/src/notebook_store.rs', 'rest-table-gateway/src/spec/openapi.rs', 'src/dag/cell_runner.rs', 'src/mcp/tests.rs', 'src/sidebar_chat/manager.rs']
Total capture bytes 12995004


In [24]:
ANALYZER = "\"\"\"Independent physical Rust syntax versus source-bound Spur graph rows.\nRun with Python + pandas + pyarrow + duckdb; all input files are local captures.\n\"\"\"\nfrom pathlib import Path\nfrom collections import Counter, defaultdict\nimport json, gzip, hashlib\nimport pandas as pd\nimport duckdb\n\nCALLABLE_KINDS={\"function\",\"method\",\"trait_method\"}\nKIND_MAP={\"trait_method\":\"method\",\"associated_constant\":\"constant\",\"trait_constant\":\"constant\",\n          \"associated_type\":\"type_alias\",\"trait_type\":\"type_alias\",\"static\":\"constant\",\n          \"foreign_static\":\"constant\",\"foreign_function\":\"function\",\"foreign_type\":\"type_alias\",\n          \"macro_definition\":\"macro\"}\n\ndef read_json(path): return json.loads(Path(path).read_text())\ndef write_json(path,obj): Path(path).write_text(json.dumps(obj,indent=2,ensure_ascii=False)+\"\\n\")\ndef normalize(name): return name.removeprefix(\"r#\")\ndef group_name(path):\n    if path.startswith(\"jute-notebook/\"):return \"Jute kernels / backend\"\n    if path.startswith(\"rest-table-gateway/\"):return \"REST table gateway\"\n    if path.startswith(\"rest-table-gateway-ext/\"):return \"Gateway extension\"\n    if path.startswith(\"crates/\"):return \"Workspace crates\"\n    if path.startswith(\"src/\"):return \"Notebook application\"\n    if path.startswith(\"tests/\"):return \"Integration tests\"\n    if path.startswith(\"vendor/\"):return \"Vendor\"\n    if path.startswith((\"docs/\",\"artifacts/\")):return \"Research / archived source\"\n    if path.startswith(\"app_gallery/\"):return \"Gallery examples\"\n    return \"Build / other\"\ndef source_class(path):\n    p=Path(path)\n    if path.startswith(\"vendor/\"):return \"vendored\"\n    if path.startswith((\"docs/\",\"artifacts/\")):return \"research_or_archived\"\n    if path.startswith(\"app_gallery/\"):return \"example\"\n    if \"tests\" in p.parts or (\"_tests.\" in p.name) or p.name==\"tests.rs\":return \"test_file\"\n    if p.name==\"build.rs\" or path.startswith(\"build_support/\"):return \"build_tool\"\n    return \"production_or_mixed\"\ndef index_symbols(rows):\n    by_key=defaultdict(list);by_end=defaultdict(list)\n    for n in rows:\n        by_key[(n[\"file_path\"],n[\"symbol_kind\"],normalize(n[\"entity_name\"]))].append(n)\n        by_end[(n[\"file_path\"],n[\"symbol_kind\"],n[\"byte_range_end\"])].append(n)\n    return by_key,by_end\ndef match_symbol(row,index):\n    by_key,by_end=index\n    kind=KIND_MAP.get(row[\"kind\"],row[\"kind\"])\n    if row[\"ident_start\"] is not None:\n        candidates=by_key.get((row[\"file_path\"],kind,normalize(row[\"name\"])),[])\n        return [n for n in candidates if\n            row[\"start\"]<=n[\"byte_range_start\"]<=row[\"ident_start\"]<row[\"ident_end\"]\n            <=n[\"byte_range_end\"]==row[\"end\"]]\n    if row[\"kind\"]==\"impl\":\n        return [n for n in by_end.get((row[\"file_path\"],\"impl\",row[\"end\"]),[])\n                if n[\"byte_range_start\"]>=row[\"start\"]]\n    # Unnamed fields/extern blocks do not get an invented name-based correspondence.\n    return []\ndef normalize_symbol(n):\n    return {\"stable_symbol_id\":n[\"stable_symbol_id\"],\"file_path\":n[\"file_path\"],\n            \"symbol_kind\":n[\"symbol_kind\"],\"entity_name\":n[\"entity_name\"],\n            \"byte_range_start\":n[\"byte_range\"][0],\"byte_range_end\":n[\"byte_range\"][1]}\n\ndef analyze(out):\n    out=Path(out);manifest=read_json(out/\"source-manifest.json\")\n    by_path={r[\"path\"]:r for r in manifest}\n    ast=read_json(out/\"rust-ast.json\")\n    parse_rows=read_json(out/\"rust-parse-results.json\")\n    parse_map={r[\"file_path\"]:r[\"status\"] for r in parse_rows}\n    published=pd.read_parquet(out/\"graph-nodes.parquet\").to_dict(\"records\")\n    fm={r[\"path\"]:r[\"content_oid\"] for r in pd.read_parquet(out/\"graph-file_manifests.parquet\").to_dict(\"records\")}\n    fresh={p:(\"unindexed\" if p not in fm else \"source_match\" if fm[p]==r[\"git_blob_oid\"] else \"stale\") for p,r in by_path.items()}\n    records=read_json(out/\"current-extractor-results.json\")\n    assert len(records)==len(manifest)\n    record_map={r[\"path\"]:r for r in records}\n    current=[];methods=[];macros=[]\n    for r in records:\n        assert r[\"source_sha256\"]==by_path[r[\"path\"]][\"sha256\"]\n        if r[\"status\"]!=\"success\":continue\n        raw=gzip.decompress((out/\"current-extractor\"/r[\"artifact\"]).read_bytes())\n        assert hashlib.sha256(raw).hexdigest()==r[\"stdout_sha256\"]\n        obj=json.loads(raw)\n        gf=obj[\"graph\"][\"file_manifests\"]\n        assert len(gf)==1 and gf[0][\"path\"]==r[\"path\"] and gf[0][\"content_oid\"]==by_path[r[\"path\"]][\"git_blob_oid\"]\n        current.extend(normalize_symbol(n) for n in obj[\"graph\"][\"symbols\"])\n        for rows,target in [(obj[\"method_sites\"][\"sites\"],methods),(obj[\"graph\"][\"macro_invocations\"][\"rows\"],macros)]:\n            for site in rows:\n                assert site[\"source_oid\"]==by_path[r[\"path\"]][\"git_blob_oid\"]\n                target.append(site)\n    pub_index=index_symbols(published);current_index=index_symbols(current)\n    symbols=[]\n    for i,r in enumerate(ast):\n        if r[\"row_type\"]!=\"symbol\":continue\n        pub=match_symbol(r,pub_index) if fresh[r[\"file_path\"]]==\"source_match\" else []\n        cur=match_symbol(r,current_index) if record_map[r[\"file_path\"]][\"status\"]==\"success\" else []\n        pub_state=fresh[r[\"file_path\"]]\n        cur_state=record_map[r[\"file_path\"]][\"status\"]\n        symbols.append({**r,\"ast_row_id\":i,\"area\":group_name(r[\"file_path\"]),\n            \"source_class\":source_class(r[\"file_path\"]),\"file_status\":pub_state,\n            \"published_status\":pub_state if pub_state!=\"source_match\" else \"represented\" if len(pub)==1 else \"ambiguous\" if pub else \"no_matching_symbol\",\n            \"published_match_count\":len(pub),\"published_symbol_id\":pub[0][\"stable_symbol_id\"] if len(pub)==1 else None,\n            \"current_status\":cur_state if cur_state!=\"success\" else \"represented\" if len(cur)==1 else \"ambiguous\" if cur else \"no_matching_symbol\",\n            \"current_match_count\":len(cur),\"current_symbol_id\":cur[0][\"stable_symbol_id\"] if len(cur)==1 else None,\n            \"symbol_context\":\"test_syntax\" if r[\"test_context\"] or source_class(r[\"file_path\"])==\"test_file\" else \"other_or_mixed_syntax\",\n            })\n    sym=pd.DataFrame(symbols)\n    for col in [\"published\",\"current\"]:\n        matches=sym[sym[col+\"_status\"]==\"represented\"][col+\"_symbol_id\"]\n        assert matches.is_unique, f\"multiple AST declarations matched one {col} symbol\"\n    site_indices={}\n    for kind,rows in [(\"method_call\",methods),(\"macro_invocation\",macros)]:\n        idx=defaultdict(list)\n        for s in rows:\n            name=s[\"method_name\"] if kind==\"method_call\" else s[\"normalized_macro_path\"]\n            idx[(s[\"file_path\"],s[\"byte_range\"][0],s[\"byte_range\"][1],normalize(name))].append(s)\n        site_indices[kind]=idx\n    calls=[]\n    for i,r in enumerate(ast):\n        if r[\"row_type\"]!=\"callsite\":continue\n        cur_state=record_map[r[\"file_path\"]][\"status\"]\n        candidates=site_indices.get(r[\"kind\"],{}).get((r[\"file_path\"],r.get(\"core_start\",r[\"start\"]),r[\"end\"],normalize(r[\"name\"])),[])\n        calls.append({**r,\"ast_row_id\":i,\"area\":group_name(r[\"file_path\"]),\n            \"published_status\":fresh[r[\"file_path\"]] if fresh[r[\"file_path\"]]!=\"source_match\" else \"no_site_table_in_published_v10\",\n            \"current_status\":cur_state if cur_state!=\"success\" else \"free_call_site_not_exposed_by_capture\" if r[\"kind\"]==\"call\" else\n                \"represented\" if len(candidates)==1 else \"ambiguous\" if candidates else \"no_matching_site\",\n            \"current_site_id\":candidates[0][\"site_id\"] if len(candidates)==1 else None,\n            \"match_method\":\"receiver_start_and_exact_end\" if r[\"kind\"]==\"method_call\" and r[\"core_start\"]!=r[\"start\"] else \"exact_source_span\",\n            \"compiler_target\":\"unknown\",\n            })\n    calls=pd.DataFrame(calls)\n    matched=calls[calls.current_status==\"represented\"].current_site_id\n    assert matched.is_unique\n    files=[]\n    sym_groups={p:g for p,g in sym.groupby(\"file_path\")}\n    call_groups={p:g for p,g in calls.groupby(\"file_path\")}\n    for r in manifest:\n        p=r[\"path\"];s=sym_groups.get(p,sym.iloc[:0]);c=call_groups.get(p,calls.iloc[:0])\n        files.append({**r,\"file_path\":p,\"area\":group_name(p),\"source_class\":source_class(p),\n            \"published_file_status\":fresh[p],\"parser_status\":parse_map[p],\n            \"current_extraction_status\":record_map[p][\"status\"],\n            \"generated_marker\":any(line.lstrip().startswith((\"//\",\"#\")) and any(tag in line.lower() for tag in (\"@generated\",\"automatically generated\",\"do not edit\")) for line in (out/\"source-snapshot\"/p).read_text(errors=\"replace\").splitlines()[:20]),\n            \"symbol_total\":len(s),\"published_symbols\":int((s.published_status==\"represented\").sum()),\n            \"current_symbols\":int((s.current_status==\"represented\").sum()),\n            \"published_gaps\":int((s.published_status==\"no_matching_symbol\").sum()),\n            \"current_gaps\":int((s.current_status==\"no_matching_symbol\").sum()),\n            \"method_calls\":int((c.kind==\"method_call\").sum()),\"macro_invocations\":int((c.kind==\"macro_invocation\").sum()),\n            \"free_calls\":int((c.kind==\"call\").sum()),\"current_site_matches\":int((c.current_status==\"represented\").sum()),\n            })\n    files=pd.DataFrame(files)\n    callable_sym=sym[sym.kind.isin(CALLABLE_KINDS)]\n    summary={\n        \"files\":len(files),\"physical_lines\":int(files.lines.sum()),\"source_bytes\":int(files.bytes.sum()),\n        \"parsed_files\":int((files.parser_status==\"parsed\").sum()),\"parse_errors\":int((files.parser_status!=\"parsed\").sum()),\n        \"published_file_states\":files.published_file_status.value_counts().to_dict(),\n        \"current_extraction_states\":files.current_extraction_status.value_counts().to_dict(),\n        \"declarations\":len(sym),\"published_declaration_states\":sym.published_status.value_counts().to_dict(),\n        \"current_declaration_states\":sym.current_status.value_counts().to_dict(),\n        \"callables\":len(callable_sym),\"published_callable_states\":callable_sym.published_status.value_counts().to_dict(),\n        \"current_callable_states\":callable_sym.current_status.value_counts().to_dict(),\n        \"ast_kind_counts\":dict(Counter(r[\"kind\"] for r in ast)),\n        \"site_states\":calls.groupby([\"kind\",\"current_status\"]).size().reset_index(name=\"count\").to_dict(\"records\"),\n        \"fresh_declarations\":int((sym.file_status==\"source_match\").sum()),\n        \"fresh_callable_declarations\":int((callable_sym.file_status==\"source_match\").sum()),\n        \"current_method_rows\":len(methods),\"current_macro_rows\":len(macros),\n        \"published_schema\":read_json(out/\"graph-manifest.json\")[\"schema_version\"],\n        \"graph_hash\":read_json(out/\"graph-manifest.json\")[\"graph_content_hash\"],\n        \"current_extractor_mode\":\"file-local syntax, project-wide target resolution not measured\",\n        \"parse_error_files\":[r for r in parse_rows if r[\"status\"]!=\"parsed\"],\n        \"current_unmeasured_files\":[r[\"path\"] for r in records if r[\"status\"]!=\"success\"],\n        \"limitations\":[\n            \"Physical syntax excludes generated/expanded macro bodies and Cargo cfg evaluation.\",\n            \"One incomplete Rust bootstrap fragment has no full-file AST denominator.\",\n            \"Stale files are not matched against current syntax.\",\n            \"Timeouts are unmeasured, not missing syntax.\",\n            \"Free-call site identity is not exposed by the snapshot capture; graph edge counts are not call-site recall.\",\n            \"Method-site presence does not certify receiver type, dispatch target, execution reachability, or panic completeness.\",\n            \"All percentages are restricted to the explicit physical syntax denominator.\",\n        ],\n    }\n    sym.to_parquet(out/\"symbol-coverage.parquet\",index=False)\n    calls.to_parquet(out/\"callsite-coverage.parquet\",index=False)\n    files.to_parquet(out/\"file-coverage.parquet\",index=False)\n    sym[sym.published_status!=\"represented\"].to_csv(out/\"published-symbol-gaps.csv\",index=False)\n    sym[sym.current_status!=\"represented\"].to_csv(out/\"current-symbol-gaps.csv\",index=False)\n    files.to_csv(out/\"file-coverage.csv\",index=False)\n    kind_counts=sym.groupby([\"kind\",\"published_status\"]).size().reset_index(name=\"count\")\n    kind_counts.to_parquet(out/\"published-kind-coverage.parquet\",index=False)\n    current_counts=sym.groupby([\"kind\",\"current_status\"]).size().reset_index(name=\"count\")\n    current_counts.to_parquet(out/\"current-kind-coverage.parquet\",index=False)\n    write_json(out/\"summary.json\",summary)\n    return summary\n\nif __name__==\"__main__\":\n    import sys\n    print(json.dumps(analyze(Path(sys.argv[1]) if len(sys.argv)>1 else Path(__file__).parent),indent=2))\n"
(OUT/'analyze.py').write_text(ANALYZER)
import importlib.util
analysis_spec=importlib.util.spec_from_file_location('rust_coverage_analysis',OUT/'analyze.py')
coverage_analysis=importlib.util.module_from_spec(analysis_spec)
analysis_spec.loader.exec_module(coverage_analysis)
coverage_summary=coverage_analysis.analyze(OUT)
print('Coverage metrics regenerated from frozen inputs. Generated marker is a comment heuristic; absence is not a handwritten-source claim.')


Coverage metrics regenerated from frozen inputs. Generated marker is a comment heuristic; absence is not a handwritten-source claim.


In [19]:
# Validate the eight attributed expressions using their exact method-token identities.
sites_df=pd.read_parquet(OUT/"callsite-coverage.parquet")
normalized=sites_df[sites_df.match_method=="receiver_start_and_exact_end"].to_dict("records")
method_gap_review=[]
for r in normalized:
    record=next(x for x in extraction_records if x["path"]==r["file_path"])
    obj=json.loads(gzip.decompress((OUT/"current-extractor"/record["artifact"]).read_bytes()))
    same=[s for s in obj["method_sites"]["sites"] if s["method_byte_range"]==[int(r["ident_start"]),int(r["ident_end"])]]
    assert len(same)==1 and same[0]["site_id"]==r["current_site_id"]
    assert same[0]["byte_range"]==[r["core_start"],r["end"]]
    method_gap_review.append({"file_path":r["file_path"],"line":r["line_start"],"ast_with_attributes":[r["start"],r["end"]],"core_call":[r["core_start"],r["end"]],"method_token":[int(r["ident_start"]),int(r["ident_end"])],"site_id":r["current_site_id"],"status":"represented_after_attribute_normalization"})
assert len(method_gap_review)==8
dump("method-gap-review.json",method_gap_review)
# Independent calibration includes nested calls with the same receiver and an attributed expression.
cal= json.loads((OUT/"calibration-ast.json").read_text())
calgraph=json.loads(gzip.decompress((OUT/"current-extractor"/calibration_record["artifact"]).read_bytes()))
cal_index=coverage_analysis.index_symbols([coverage_analysis.normalize_symbol(n) for n in calgraph["graph"]["symbols"]])
cal_results=[]
for r in cal:
    if r["row_type"]=="symbol":
        candidates=coverage_analysis.match_symbol(r,cal_index)
        expected=r["kind"] not in {"tuple_field","foreign_function","extern_block"}
        assert (len(candidates)==1)==expected,(r,candidates)
        cal_results.append({"kind":r["kind"],"name":r["name"],"represented":len(candidates)==1})
method_ids=[]
for r in cal:
    if r["kind"]=="method_call":
        matches=[s for s in calgraph["method_sites"]["sites"] if s["byte_range"]==[r["core_start"],r["end"]] and s["method_byte_range"]==[r["ident_start"],r["ident_end"]]]
        assert len(matches)==1
        method_ids.append(matches[0]["site_id"])
assert len(set(method_ids))==3
dump("calibration-verification.json",{"symbol_results":cal_results,"distinct_method_sites":len(set(method_ids)),"normalized_project_sites":len(method_gap_review),"status":"passed"})
print("Calibration and eight attributed method correspondences passed; all matched site IDs are distinct.")


Calibration and eight attributed method correspondences passed; all matched site IDs are distinct.


## How to read the coverage

**Published v10** is the graph currently on disk. Matching is permitted only when its file content OID equals the captured source OID. A stale symbol remains unverified even if its name still exists.

**Current extractor** is a separate experiment: one physical file per isolated extraction root, using the tested detector binary. It measures syntax collection and preserves method/macro invocation identities. It does not evaluate project-wide call-target resolution. Six files exceeded the 30-second per-file budget and remain unmeasured.

The baseline includes physical declarations under all `cfg` branches and treats macro token trees as opaque. One include-style Rust fragment cannot be parsed as a full module and is excluded from symbol/call denominators. Fields without names and extern blocks are shown explicitly; they are not discarded to inflate coverage.

An initial comparison found eight method-range differences caused by expression attributes. Matching now uses the AST receiver start, exact call end, and method name. The same method-token ranges validate that correspondence.

In [40]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import io
files_df=pd.read_parquet(OUT/"file-coverage.parquet")
sym_df=pd.read_parquet(OUT/"symbol-coverage.parquet")
call_df=pd.read_parquet(OUT/"callsite-coverage.parquet")
summary=json.loads((OUT/"summary.json").read_text())
COLORS={"represented":"#237a57","source_match":"#237a57","stale":"#bc7b16","unindexed":"#ae414c","no_matching_symbol":"#5544a8","timeout":"#9babc0","free_call_site_not_exposed_by_capture":"#64748b"}
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":10,"axes.spines.top":False,"axes.spines.right":False,"svg.fonttype":"none","figure.facecolor":"white","axes.labelcolor":"#233044","text.color":"#233044","xtick.color":"#475569","ytick.color":"#475569"})
fig,axes=plt.subplots(2,2,figsize=(16,11.5),gridspec_kw={"height_ratios":[1,1.35]})
fig.suptitle("Rust syntax represented by Spur graph · spur-notebook",x=.02,ha="left",y=.99,fontsize=21,fontweight="bold")
fig.text(.02,.946,"689 captured Rust files · independent syn AST · published v10 versus current file-local extractor",fontsize=11,color="#526175")
def stack(ax,frame,column,order,title):
    grouped=frame.groupby(column).size()
    left=0
    for status in order:
        n=int(grouped.get(status,0))
        if not n:continue
        ax.barh([0],[n],left=left,color=COLORS[status],height=.5,label=f'{status.replace("_"," ")} · {n:,}')
        if n/len(frame)>.06:ax.text(left+n/2,0,f"{n:,}\n{n/len(frame):.1%}",ha="center",va="center",color="white",fontweight="bold")
        left+=n
    ax.set_xlim(0,len(frame));ax.set_yticks([]);ax.set_xlabel(f"Out of {len(frame):,} "+("Rust files" if column=="published_file_status" else "callable declarations"))
    ax.set_title(title,loc="left",pad=14,fontweight="bold")
    ax.legend(loc="upper left",bbox_to_anchor=(0,-.28),ncol=2,frameon=False,fontsize=9)
stack(axes[0,0],files_df,"published_file_status",["source_match","stale","unindexed"],"Published graph: file freshness")
callables=sym_df[sym_df.kind.isin(["function","method","trait_method"])]
stack(axes[0,1],callables,"published_status",["represented","stale","unindexed"],"Published graph: function / method declarations")
area=sym_df.groupby(["area","published_status"]).size().unstack(fill_value=0)
area=area.loc[area.sum(axis=1).sort_values(ascending=True).index]
left=[0.0]*len(area)
for status in ["represented","stale","unindexed","no_matching_symbol"]:
    vals=area.get(status,pd.Series(0,index=area.index))/area.sum(axis=1)
    axes[1,0].barh(area.index,vals,left=left,label=status.replace("_"," "),color=COLORS[status])
    left=[a+b for a,b in zip(left,vals)]
axes[1,0].xaxis.set_major_formatter(PercentFormatter(1));axes[1,0].set_xlim(0,1)
axes[1,0].set_title("Published graph: declaration coverage by source area",loc="left",fontweight="bold",pad=14)
axes[1,0].set_xlabel("Share of parsed declarations; includes fields and containers")
axes[1,0].legend(loc="upper left",bbox_to_anchor=(0,-.18),ncol=2,frameon=False,fontsize=9)
site=call_df[call_df.kind.isin(["method_call","macro_invocation"])]
site=site.groupby(["kind","current_status"]).size().unstack(fill_value=0)
left=[0]*len(site)
for status in ["represented","timeout"]:
    vals=site.get(status,pd.Series(0,index=site.index)).tolist()
    axes[1,1].barh(["Macro invocations" if k=="macro_invocation" else "Method invocations" for k in site.index],vals,left=left,color=COLORS[status],label="exact site match" if status=="represented" else "30s timeout: unmeasured")
    for i,(start,v) in enumerate(zip(left,vals)):
        if v:
            narrow=v<max(left+vals)*.12
            axes[1,1].text(start+v+800 if narrow else start+v/2,i,f"{v:,}",ha="left" if narrow else "center",va="center",color="#475569" if narrow else "white",fontweight="bold")
    left=[a+b for a,b in zip(left,vals)]
axes[1,1].set_xlim(0,max(left)*1.06)
axes[1,1].set_title("Current extractor: exact source site identities",loc="left",fontweight="bold",pad=14)
axes[1,1].set_xlabel("Physical AST invocations; macro expansion stays opaque")
axes[1,1].legend(loc="upper left",bbox_to_anchor=(0,-.18),frameon=False,fontsize=9)
fig.text(.02,.018,"Scope: 688 complete Rust files parsed; one include fragment excluded from AST denominators. Six current-extractor files timed out.\nRepresentation does not establish call-target correctness, branch feasibility, execution coverage, or absence of panics.",fontsize=10,color="#526175")
fig.subplots_adjust(top=.88,bottom=.17,hspace=.87,wspace=.53,left=.19,right=.98)
fig.savefig(OUT/"coverage-overview.svg",bbox_inches="tight")
fig.savefig(OUT/"coverage-overview.png",dpi=150,bbox_inches="tight")
plt.close(fig)
# Detailed declaration-kind comparison uses the same denominator in both panels.
kinds=sym_df.groupby("kind").size().sort_values().index
fig,axs=plt.subplots(1,2,figsize=(15,10),sharey=True)
for ax,column,title,statuses in [
    (axs[0],"published_status","Published graph · source-hash gated",["represented","stale","unindexed","no_matching_symbol"]),
    (axs[1],"current_status","Current extractor · file-local",["represented","timeout","no_matching_symbol"])]:
    data=sym_df.groupby(["kind",column]).size().unstack(fill_value=0).reindex(kinds)
    left=[0.0]*len(data)
    for status in statuses:
        vals=data.get(status,pd.Series(0,index=kinds))/data.sum(axis=1)
        ax.barh([k.replace("_"," ") for k in kinds],vals,left=left,color=COLORS[status],label=status.replace("_"," "))
        left=[a+b for a,b in zip(left,vals)]
    ax.set_xlim(0,1);ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.set_title(title,loc="left",fontweight="bold");ax.legend(loc="upper left",bbox_to_anchor=(0,-.07),frameon=False,fontsize=9,ncol=2)
axs[0].set_ylabel("Physical AST declaration kind")
fig.suptitle("Where declaration coverage ends",fontsize=20,x=.02,ha="left",fontweight="bold")
fig.text(.02,.005,"Tuple fields are present inside their parent type's bytes, but lack individual symbol nodes. Foreign declarations lack symbol nodes.\nA gray segment means extraction exceeded its time budget, not that the source syntax is unsupported.",fontsize=10,color="#526175")
fig.subplots_adjust(left=.18,top=.9,bottom=.16,wspace=.12)
fig.savefig(OUT/"coverage-by-kind.svg",bbox_inches="tight")
fig.savefig(OUT/"coverage-by-kind.png",dpi=150,bbox_inches="tight")
plt.close(fig)
for svg_name in ["coverage-overview.svg","coverage-by-kind.svg"]:
    svg_path=OUT/svg_name
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines())+"\n")
compact_symbols=sym_df[["file_path","kind","name","line_start","line_end","published_status","current_status"]].values.tolist()
payload={"symbols_json":json.dumps(compact_symbols),"summary_json":json.dumps(summary),"files_json":files_df.to_json(orient="records"),"overview_svg":(OUT/"coverage-overview.svg").read_text(),"kinds_svg":(OUT/"coverage-by-kind.svg").read_text()}
spur.put("rust_coverage_bundle",pd.DataFrame([payload]))
print("Exported SVG and PNG overview / kind charts; published rust_coverage_bundle (one Arrow port).")


symbols_json,summary_json,files_json,overview_svg,kinds_svg


Exported SVG and PNG overview / kind charts; published rust_coverage_bundle (one Arrow port).


In [21]:
SELECT kind, published_status, count
FROM read_parquet('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-rust-graph-coverage/published-kind-coverage.parquet')
ORDER BY kind, published_status

kind,published_status,count
associated_constant,represented,14
associated_type,represented,32
constant,represented,776
constant,stale,25
constant,unindexed,10
enum,represented,371
enum,stale,30
enum,unindexed,20
enum_variant,represented,1590
enum_variant,stale,259


In [38]:
SELECT file_path, area, source_class, parser_status, published_file_status,
       symbol_total, published_symbols, published_gaps,
       current_extraction_status, current_symbols, current_gaps,
       lines, method_calls, macro_invocations
FROM read_parquet('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-rust-graph-coverage/file-coverage.parquet')
ORDER BY (published_file_status != 'source_match') DESC, symbol_total DESC

file_path,area,source_class,parser_status,published_file_status,symbol_total,published_symbols,published_gaps,current_extraction_status,current_symbols,current_gaps,lines,method_calls,macro_invocations
jute-notebook/src-tauri/src/commands.rs,Jute kernels / backend,production_or_mixed,parsed,stale,1039,0,0,timeout,0,0,8361,1301,556
src/dag/engine.rs,Notebook application,production_or_mixed,parsed,stale,315,0,0,success,309,6,3879,905,184
src/mcp/daemon/control.rs,Notebook application,production_or_mixed,parsed,stale,238,0,0,success,237,1,4138,930,69
jute-notebook/src-tauri/src/state.rs,Jute kernels / backend,production_or_mixed,parsed,stale,142,0,0,success,141,1,1377,363,69
crates/spurlab-telemetry/src/service.rs,Workspace crates,production_or_mixed,parsed,unindexed,136,0,0,success,134,2,1651,536,171
crates/spurlab-telemetry/src/events.rs,Workspace crates,production_or_mixed,parsed,unindexed,117,0,0,success,117,0,654,55,20
src/telemetry.rs,Notebook application,production_or_mixed,parsed,unindexed,110,0,0,success,105,5,856,289,87
src/commands/telemetry.rs,Notebook application,production_or_mixed,parsed,unindexed,85,0,0,success,82,3,959,323,121
jute-notebook/src-tauri/src/backend/commands.rs,Jute kernels / backend,production_or_mixed,parsed,stale,76,0,0,success,64,12,639,87,31
crates/spurlab-telemetry/src/buffer.rs,Workspace crates,production_or_mixed,parsed,unindexed,74,0,0,success,73,1,718,119,119


In [39]:
SELECT file_path, line_start, line_end, kind, name, symbol_context,
       published_status, current_status, published_symbol_id, current_symbol_id,
       start AS source_byte_start, "end" AS source_byte_end
FROM read_parquet('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-rust-graph-coverage/symbol-coverage.parquet')
WHERE published_status != 'represented' OR current_status != 'represented'
ORDER BY file_path, source_byte_start

file_path,line_start,line_end,kind,name,symbol_context,published_status,current_status,published_symbol_id,current_symbol_id,source_byte_start,source_byte_end
crates/spurlab-telemetry/build.rs,1,12,function,main,other_or_mixed_syntax,unindexed,represented,NaN,c653b4d0e0bb7a7c,0,666
crates/spurlab-telemetry/build.rs,14,58,module,tests,test_syntax,unindexed,represented,NaN,7af39cf70925c73f,668,2376
crates/spurlab-telemetry/build.rs,16,20,function,build_entry,test_syntax,unindexed,represented,NaN,c406d86c85e621b2,697,797
crates/spurlab-telemetry/build.rs,22,57,function,personal_keys_fail_before_compilation_without_echoing_configuration,test_syntax,unindexed,represented,NaN,bc821f2305d00f98,803,2374
crates/spurlab-telemetry/src/buffer.rs,15,16,constant,BATCH_CAPACITY,other_or_mixed_syntax,unindexed,represented,NaN,559b43bf9747d36a,741,858
crates/spurlab-telemetry/src/buffer.rs,18,36,struct,OperationTicket,other_or_mixed_syntax,unindexed,represented,NaN,73723ca520e2fc58,860,1493
crates/spurlab-telemetry/src/buffer.rs,32,32,field,authority,other_or_mixed_syntax,unindexed,represented,NaN,51cbed95415618ff,1400,1418
crates/spurlab-telemetry/src/buffer.rs,33,33,field,epoch,other_or_mixed_syntax,unindexed,represented,NaN,4eb729bfeadfba38,1424,1434
crates/spurlab-telemetry/src/buffer.rs,34,34,field,started_allowed,other_or_mixed_syntax,unindexed,represented,NaN,9d00d09604fa9b39,1440,1461
crates/spurlab-telemetry/src/buffer.rs,35,35,field,terminal_consumed,other_or_mixed_syntax,unindexed,represented,NaN,f6caf9e246d8343a,1467,1490


In [41]:
# Self-contained scientific report: charts stay visible when scripts are disabled.
from IPython.display import HTML, display
import html as html_lib
bundle=spur.get("rust_coverage_bundle")
if hasattr(bundle,"to_pandas"): bundle=bundle.to_pandas()
if isinstance(bundle,pd.DataFrame): bundle=bundle.iloc[0].to_dict()
elif isinstance(bundle,list): bundle=bundle[0]
assert isinstance(bundle,dict),type(bundle)
s=json.loads(bundle["summary_json"]);frows=json.loads(bundle["files_json"]);srows=json.loads(bundle["symbols_json"])
def h(x):return html_lib.escape(str(x),quote=True)
def svg_body(x):return x[x.index("<svg"):x.rindex("</svg>")+6]
def file_row(r,i):
    return f'<tr data-index="{i}" tabindex="0"><td><button class="filelink" data-file="{i}">{h(r["file_path"])}</button></td><td>{h(r["source_class"])}</td><td>{h(r["published_file_status"])}</td><td class="num">{r["published_symbols"]:,} / {r["symbol_total"]:,}</td><td>{h(r["current_extraction_status"])}</td><td class="num">{r["current_symbols"]:,}</td></tr>'
table="".join(file_row(r,i) for i,r in enumerate(frows))
legend='<span><i style="background:#237a57"></i>Source matches</span><span><i style="background:#bc7b16"></i>Stale</span><span><i style="background:#ae414c"></i>Unindexed</span>'
heat=[];y=4
for area in sorted({r["area"] for r in frows}):
    ids=[i for i,r in enumerate(frows) if r["area"]==area]
    heat.append(f'<text x="0" y="{y+14}" font-size="12" fill="#334155">{h(area)} ({len(ids)})</text>')
    for j,i in enumerate(ids):
        r=frows[i];x=224+(j%36)*19;ty=y+(j//36)*19
        color={"source_match":"#237a57","stale":"#bc7b16","unindexed":"#ae414c"}[r["published_file_status"]]
        heat.append(f'<rect x="{x}" y="{ty}" width="15" height="15" rx="2" fill="{color}" class="tile" data-file="{i}" tabindex="0" role="button" aria-label="{h(r["file_path"])}"><title>{h(r["file_path"])} · {h(r["published_file_status"])} · {r["published_symbols"]}/{r["symbol_total"]} declarations</title></rect>')
    y+=max(1,(len(ids)+35)//36)*19+18
heatmap=f'<svg id="filemap" viewBox="0 0 920 {y}" role="img" aria-label="Each square is one captured Rust file">{"".join(heat)}</svg>'
classes="".join(f'<option value="{h(c)}">{h(c.replace("_"," "))}</option>' for c in sorted({r["source_class"] for r in frows}))
payload=json.dumps({"files":frows,"symbols":srows},ensure_ascii=True,separators=(",",":")).replace("<","\\u003c")
css="""
:root{color-scheme:light;font-family:system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;color:#233044;background:#f5f7fa}
*{box-sizing:border-box}body{margin:0}main{max-width:1240px;margin:auto;padding:32px 28px 64px}
h1{font-size:34px;letter-spacing:-.8px;margin:6px 0 10px}h2{font-size:20px;margin:0 0 12px}h3{font-size:16px;margin:16px 0 8px}
p{line-height:1.55;margin:8px 0 16px}.kicker{font-size:12px;letter-spacing:1.2px;color:#64748b;text-transform:uppercase}.sub{color:#526175;max-width:1000px}
.metrics{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin:26px 0}.metric{padding:18px;border:1px solid #dce3eb;background:white;border-radius:6px}.metric strong{font-size:28px;display:block;font-variant-numeric:tabular-nums}.metric span{font-size:12px;color:#526175}.metric b{font-size:13px;display:block;margin:5px 0;font-weight:600}
section{margin:24px 0;background:white;border:1px solid #dce3eb;border-radius:6px;padding:24px}
svg{max-width:100%;height:auto}.figure svg{display:block;width:100%}.legend{display:flex;gap:18px;flex-wrap:wrap;font-size:12px;color:#526175;margin:12px 0}.legend span{display:flex;align-items:center;gap:6px}.legend i{width:12px;height:12px;border-radius:2px;display:inline-block}
.tile{cursor:pointer}.tile:hover,.tile:focus{stroke:#0f172a;stroke-width:3;outline:none}
.controls{display:flex;gap:12px;align-items:end;flex-wrap:wrap;margin:18px 0}label{font-size:12px;display:flex;gap:5px;flex-direction:column;color:#475569}input,select,button{font:inherit}input,select{padding:9px 10px;border:1px solid #b9c5d3;border-radius:4px;background:white;color:#233044}
input{width:290px}button{cursor:pointer}button.filelink{border:0;padding:0;background:none;color:#214f7c;font:inherit;text-align:left;overflow-wrap:anywhere}button.filelink:hover{text-decoration:underline}
.scroll{max-height:440px;overflow:auto;border:1px solid #e2e8f0}table{border-collapse:collapse;width:100%;font-size:12px;text-align:left}th{position:sticky;top:0;background:#f1f5f9;z-index:1;white-space:nowrap;font-weight:600}td,th{padding:9px 10px;border-bottom:1px solid #e5ebf1;vertical-align:top}tr:hover{background:#f4f8fc}.num{font-family:ui-monospace,monospace;text-align:right;white-space:nowrap}code{font-family:ui-monospace,monospace;font-size:.9em;overflow-wrap:anywhere}.note{font-size:12px;color:#526175}.detail{background:#f7f9fc;padding:16px;border:1px solid #e2e8f0;border-radius:4px;margin:16px 0}.detail h3{overflow-wrap:anywhere}.detail table{background:white}.detail .scroll{max-height:300px}details summary{cursor:pointer;font-weight:600}
#rowcount{font-size:12px;color:#64748b}#selected-status{font-size:13px}#symbol-strip{width:100%;height:94px}
footer{font-size:12px;color:#64748b;line-height:1.7}
@media(max-width:800px){main{padding:20px 14px}.metrics{grid-template-columns:repeat(2,1fr)}section{padding:16px}h1{font-size:28px}.scroll{max-height:360px}}
@media print{body{background:white}main{padding:0}.controls{display:none}.scroll{max-height:none;overflow:visible}.metric,section{break-inside:avoid}}
"""
js=r"""
const data=JSON.parse(document.getElementById("coverage-data").textContent);
const esc=x=>String(x).replace(/[&<>"']/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;","\"":"&quot;","'":"&#39;"}[c]));
const colors={represented:"#237a57",stale:"#bc7b16",unindexed:"#ae414c",no_matching_symbol:"#5544a8",timeout:"#9babc0",ambiguous:"#a855f7"};
const symbolMap=new Map();
for(const row of data.symbols){if(!symbolMap.has(row[0]))symbolMap.set(row[0],[]);symbolMap.get(row[0]).push(row);}
function selectFile(i){
 const f=data.files[i], syms=symbolMap.get(f.file_path)||[];
 document.getElementById("selected-name").textContent=f.file_path;
 document.getElementById("selected-status").textContent=f.lines.toLocaleString()+" physical lines · Published: "+f.published_file_status+" ("+f.published_symbols+"/"+f.symbol_total+" declaration nodes verified) · Current extraction: "+f.current_extraction_status+" ("+f.current_symbols+"/"+f.symbol_total+" represented) · Parser: "+f.parser_status;
 document.getElementById("selected-hash").textContent="Source SHA256 "+f.sha256;
 const denom=Math.max(f.lines,1);let drawing='<text x="0" y="12" font-size="11" fill="#526175">Declaration spans along physical file lines 1–'+f.lines+'. Containers overlap; this is not line execution coverage.</text>';
 for(const [statusIndex,y,label] of [[5,24,"Published"],[6,56,"Current"]]){
  drawing+='<text x="0" y="'+(y+11)+'" font-size="11" fill="#526175">'+label+'</text><rect x="78" y="'+y+'" width="900" height="16" fill="#e8edf3"/>';
  const ordered=[...syms].sort((a,b)=>(a[statusIndex]==="represented"?-1:1)-(b[statusIndex]==="represented"?-1:1));
  for(const r of ordered){
   const x=78+900*(r[3]-1)/denom,w=Math.max(1,900*(r[4]-r[3]+1)/denom);
   drawing+='<rect x="'+x+'" y="'+y+'" width="'+w+'" height="16" fill="'+(colors[r[statusIndex]]||"#64748b")+'" opacity=".8"><title>'+esc(r[1])+" "+esc(r[2])+" · lines "+r[3]+"–"+r[4]+" · "+esc(r[statusIndex])+"</title></rect>";
  }
 }
 document.getElementById("symbol-strip").innerHTML=drawing;
 document.getElementById("symbol-body").innerHTML=syms.map(r=>"<tr><td>"+esc(r[1])+"</td><td><code>"+esc(r[2])+'</code></td><td class="num">'+r[3]+"–"+r[4]+"</td><td>"+esc(r[5])+"</td><td>"+esc(r[6])+"</td></tr>").join("");
}
function filterFiles(){
 const q=document.getElementById("search").value.toLowerCase(),cls=document.getElementById("class-filter").value,state=document.getElementById("state-filter").value;let n=0;
 for(const row of document.querySelectorAll("#file-body tr")){
  const f=data.files[Number(row.dataset.index)],ok=(!q||f.file_path.toLowerCase().includes(q))&&(!cls||f.source_class===cls)&&(!state||f.published_file_status===state);
  row.hidden=!ok;if(ok)n++;
 }
 for(const tile of document.querySelectorAll(".tile")){
  const f=data.files[Number(tile.dataset.file)],ok=(!q||f.file_path.toLowerCase().includes(q))&&(!cls||f.source_class===cls)&&(!state||f.published_file_status===state);
  tile.style.opacity=ok?"1":".12";
 }
 document.getElementById("rowcount").textContent=n+" / "+data.files.length+" captured Rust files shown";
}
document.addEventListener("click",e=>{const el=e.target.closest("[data-file]");if(el)selectFile(Number(el.dataset.file));});
document.getElementById("filemap").addEventListener("keydown",e=>{if((e.key==="Enter"||e.key===" ")&&e.target.dataset.file!==undefined){e.preventDefault();selectFile(Number(e.target.dataset.file));}});
for(const id of ["search","class-filter","state-filter"])document.getElementById(id).addEventListener("input",filterFiles);
filterFiles();selectFile(Math.max(0,data.files.findIndex(f=>f.published_file_status==="unindexed"&&f.symbol_total>100)));
"""
site_count=lambda kind,status:sum(r["count"] for r in s["site_states"] if r["kind"]==kind and r["current_status"]==status)
method_count=site_count("method_call","represented");method_unknown=site_count("method_call","timeout")
macro_count=site_count("macro_invocation","represented");macro_unknown=site_count("macro_invocation","timeout")
html_doc=f"""<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Rust source and Spur graph coverage</title><style>{css}</style></head><body><main>
<header><div class="kicker">Measured source comparison · 02 November 2026</div><h1>Rust source and Spur graph coverage</h1><p class="sub">Physical Rust declarations and invocation sites in <code>spur-notebook</code>, independently parsed with <code>syn 2.0.118</code>. The published graph is source-hash gated; current-extractor results come from separate file-local captures.</p></header>
<div class="metrics">
<div class="metric"><strong>{s["files"]:,}</strong><b>captured Rust files</b><span>{s["parsed_files"]:,} complete modules parsed; {s["parse_errors"]} include fragment excluded</span></div>
<div class="metric"><strong>{s["published_callable_states"]["represented"]:,} / {s["callables"]:,}</strong><b>callable declarations verified in published graph</b><span>{s["published_callable_states"]["represented"]/s["callables"]:.1%} of parsed physical declarations; {s["published_callable_states"].get("stale",0):,} stale, {s["published_callable_states"].get("unindexed",0):,} unindexed</span></div>
<div class="metric"><strong>{method_count:,}</strong><b>method sites matched in current captures</b><span>{method_unknown:,} further sites unmeasured in {s["current_extraction_states"].get("timeout",0)} timed-out files</span></div>
<div class="metric"><strong>{macro_count:,}</strong><b>macro sites matched in current captures</b><span>{macro_unknown:,} further sites unmeasured; expansion remains opaque</span></div>
</div>
<section class="figure"><h2>Coverage overview</h2>{svg_body(bundle["overview_svg"])}</section>
<section><h2>Each square is one physical Rust file</h2><p class="note">Color shows published graph freshness. Click a square or filename to inspect declaration kinds and source line spans. Tests, examples, vendored code and research copies remain separate filters.</p><div class="legend">{legend}</div>{heatmap}
<div class="controls"><label>Find a file<input id="search" type="search" placeholder="e.g. telemetry or state.rs" autocomplete="off"></label><label>Source category<select id="class-filter"><option value="">All categories</option>{classes}</select></label><label>Published graph<select id="state-filter"><option value="">All source states</option><option value="source_match">Source matches</option><option value="stale">Stale</option><option value="unindexed">Unindexed</option></select></label><span id="rowcount" aria-live="polite">{len(frows)} captured files</span></div>
<div id="file-detail" class="detail"><h3 id="selected-name">Select a file to inspect its declaration spans</h3><p id="selected-status">The file table below remains available when scripts are disabled.</p><code id="selected-hash"></code><svg id="symbol-strip" viewBox="0 0 1000 94" aria-label="Declaration spans in selected source file"></svg><div class="scroll"><table><thead><tr><th>Kind</th><th>Name</th><th>Source lines</th><th>Published</th><th>Current extraction</th></tr></thead><tbody id="symbol-body"></tbody></table></div></div>
<div class="scroll"><table><thead><tr><th>Rust file</th><th>Source category</th><th>Published file</th><th>Verified / declarations</th><th>Current extraction</th><th>Current declarations</th></tr></thead><tbody id="file-body">{table}</tbody></table></div></section>
<section class="figure"><details><summary>Declaration coverage by kind</summary>{svg_body(bundle["kinds_svg"])}</details></section>
<section><h2>Where the evidence stops</h2><p>On matching source files, every parsed function, method and trait-method declaration has a symbol node. Unnamed tuple fields and foreign declarations lack individual nodes; their parent text can still be present. The current capture exposes method and macro sites, while free-call invocation identity is not measured by this capture API.</p><p>{s["current_extraction_states"].get("timeout",0)} current-extractor files exceeded a 30-second budget. They account for {s["current_declaration_states"].get("timeout",0):,} declarations, {method_unknown:,} method sites and {macro_unknown:,} macro sites; gray means unmeasured. One incomplete bootstrap fragment has no full-module AST denominator.</p><p>Macro-generated code, Cargo feature selection, trait dispatch, implicit compiler checks, branch feasibility and runtime reachability need separate evidence. This report does not establish absence of panics.</p></section>
<footer>Published schema: <code>{h(s["published_schema"])}</code> · Graph hash: <code>{h(s["graph_hash"])}</code><br>All symbol matches require source identity, declaration kind, name, identifier containment and exact declaration end. Attribute-bearing method expressions normalize to their receiver start and retain exact method-token/end positions.<br>Generated-source markers are a comment heuristic; no marker does not prove handwritten origin. Physical line totals include comments and whitespace.</footer>
<script id="coverage-data" type="application/json">{payload}</script><script>{js}</script></main></body></html>"""
(OUT/"coverage.html").write_text(html_doc)
(OUT/"dashboard.js").write_text(js)
assert len(frows)==s["files"] and len(srows)==s["declarations"]
display(HTML(html_doc))


In [27]:
import importlib.util
print("Interactive render checks",{"playwright":importlib.util.find_spec("playwright") is not None,"node":shutil.which("node"),"chromium":shutil.which("chromium"),"chrome":Path("/Applications/Google Chrome.app/Contents/MacOS/Google Chrome").exists()})
print("Saved HTML bytes",(OUT/"coverage.html").stat().st_size)
print("Generated-marker files",pd.read_parquet(OUT/"file-coverage.parquet").query("generated_marker").file_path.tolist())


Interactive render checks {'playwright': False, 'node': '/opt/homebrew/bin/node', 'chromium': None, 'chrome': True}
Saved HTML bytes 3671735
Generated-marker files []


In [28]:
# Verify declarations that have no supported correspondence, without inventing names.
from collections import defaultdict
sym_df=pd.read_parquet(OUT/"symbol-coverage.parquet")
published_nodes=pd.read_parquet(OUT/"graph-nodes.parquet").to_dict("records")
current_nodes=[]
for record in extraction_records:
    if record["status"]=="success":
        o=json.loads(gzip.decompress((OUT/"current-extractor"/record["artifact"]).read_bytes()))
        current_nodes.extend(coverage_analysis.normalize_symbol(n) for n in o["graph"]["symbols"])
def check_unmatched(rows,node_rows,status_column):
    idx=defaultdict(list)
    for n in node_rows:idx[n["file_path"]].append(n)
    results=[]
    for r in rows[rows[status_column]=="no_matching_symbol"].to_dict("records"):
        if r["kind"]=="tuple_field":
            candidates=[n for n in idx[r["file_path"]] if n["symbol_kind"]=="field" and n["byte_range_start"]>=r["start"] and n["byte_range_end"]<=r["end"]]
        elif r["kind"]=="extern_block":
            candidates=[n for n in idx[r["file_path"]] if n["byte_range_start"]==r["start"] and n["byte_range_end"]==r["end"]]
        else:
            candidates=[n for n in idx[r["file_path"]] if n["symbol_kind"]=="function" and n["byte_range_start"]<=r["ident_start"]<n["byte_range_end"]<=r["end"]]
        assert not candidates,(r,candidates)
        results.append({"file_path":r["file_path"],"kind":r["kind"],"start":r["start"],"end":r["end"],"matching_individual_nodes":len(candidates)})
    return results
unsupported={"published":check_unmatched(sym_df,published_nodes,"published_status"),"current":check_unmatched(sym_df,current_nodes,"current_status"),
 "interpretation":"No individual tuple-field / FFI node found at these source locations; enclosing type or block text may still be stored."}
dump("unsupported-node-check.json",unsupported)
print("Direct node checks", {k:len(unsupported[k]) for k in ["published","current"]})
# Read the actual review issue's durable audits; no private solver storage is accessed.
import sqlite3
dbs=sorted(Path("/Users/kevintruong/project/SpurCode/.beads").glob("*.db"))
assert dbs
with sqlite3.connect(dbs[0].as_uri()+"?mode=ro",uri=True) as db:
    db.row_factory=sqlite3.Row
    audits=[dict(r) for r in db.execute("SELECT * FROM comments WHERE issue_id=? ORDER BY id",("bd-l8h",))]
dump("worker-tracker-audits.json",audits)
print("Reviewer tracker comments",[(r.get("id"),list(r.keys())) for r in audits][-5:])
print("MCP artifact HTML is self-contained. Browser binary is available for an offline smoke check.")


Direct node checks {'published': 376, 'current': 406}
Reviewer tracker comments [(2012, ['id', 'issue_id', 'author', 'text', 'created_at']), (2013, ['id', 'issue_id', 'author', 'text', 'created_at']), (2014, ['id', 'issue_id', 'author', 'text', 'created_at']), (2015, ['id', 'issue_id', 'author', 'text', 'created_at']), (2016, ['id', 'issue_id', 'author', 'text', 'created_at'])]
MCP artifact HTML is self-contained. Browser binary is available for an offline smoke check.


In [43]:
# Offline browser smoke check in a fresh profile, with no user browser state.
import asyncio, urllib.request, base64, uuid
from tornado.websocket import websocket_connect
CHROME="/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"
PROFILE=PRIVATE/("chrome-profile-"+str(uuid.uuid4()))
chrome_log=open(OUT/"browser.log","wb")
chrome_proc=subprocess.Popen([CHROME,"--headless=new","--remote-debugging-port=0","--user-data-dir="+str(PROFILE),"--disable-background-networking","--no-first-run","--no-default-browser-check","--disable-extensions","about:blank"],stdout=chrome_log,stderr=chrome_log)
browser_results={};ws=None
try:
    for attempt in range(100):
        if (PROFILE/"DevToolsActivePort").exists():break
        if chrome_proc.poll() is not None:raise RuntimeError("Chrome exited; inspect browser.log")
        await asyncio.sleep(.1)
    port=int((PROFILE/"DevToolsActivePort").read_text().splitlines()[0])
    request=urllib.request.Request("http://127.0.0.1:"+str(port)+"/json/new?about:blank",method="PUT")
    tab=json.load(urllib.request.urlopen(request))
    ws=await websocket_connect(tab["webSocketDebuggerUrl"])
    cdp_id=0
    async def cdp(method,params=None):
        global cdp_id
        cdp_id+=1;cid=cdp_id
        await ws.write_message(json.dumps({"id":cid,"method":method,"params":params or {}}))
        while True:
            message=json.loads(await asyncio.wait_for(ws.read_message(),10))
            if message.get("id")==cid:
                assert "error" not in message,message
                return message.get("result",{})
    async def evaluate(code):
        result=await cdp("Runtime.evaluate",{"expression":code,"returnByValue":True,"awaitPromise":True})
        assert "exceptionDetails" not in result,result
        return result["result"].get("value")
    await cdp("Page.enable");await cdp("Runtime.enable")
    await cdp("Emulation.setDeviceMetricsOverride",{"width":1440,"height":1100,"deviceScaleFactor":1,"mobile":False})
    await cdp("Page.navigate",{"url":(OUT/"coverage.html").as_uri()})
    for attempt in range(100):
        if await evaluate("document.readyState==='complete' && typeof selectFile==='function'"):break
        await asyncio.sleep(.1)
    assert await evaluate("typeof selectFile==='function'")
    browser_results["initial_visible_files"]=await evaluate("document.querySelectorAll('#file-body tr:not([hidden])').length")
    assert browser_results["initial_visible_files"]==689
    await evaluate("document.getElementById('state-filter').value='unindexed';filterFiles()")
    browser_results["unindexed_filter"]=await evaluate("document.querySelectorAll('#file-body tr:not([hidden])').length")
    assert browser_results["unindexed_filter"]==21
    await evaluate("document.getElementById('state-filter').value='';document.getElementById('class-filter').value='test_file';filterFiles()")
    browser_results["test_file_filter"]=await evaluate("document.querySelectorAll('#file-body tr:not([hidden])').length")
    assert browser_results["test_file_filter"]==sum(r["source_class"]=="test_file" for r in frows)
    await evaluate("document.getElementById('class-filter').value='';document.getElementById('search').value='__no_such_rust_source__';filterFiles()")
    browser_results["empty_search"]=await evaluate("document.querySelectorAll('#file-body tr:not([hidden])').length")
    assert browser_results["empty_search"]==0
    await evaluate("document.getElementById('search').value='';filterFiles();document.querySelector('.tile').dispatchEvent(new KeyboardEvent('keydown',{key:'Enter',bubbles:true}))")
    first_tile_index=await evaluate("Number(document.querySelector('.tile').dataset.file)")
    assert await evaluate("document.getElementById('selected-name').textContent")==frows[first_tile_index]["file_path"]
    browser_results["keyboard_selection"]="passed"
    index=next(i for i,r in enumerate(frows) if r["current_gaps"]>0 and r["published_file_status"]=="source_match")
    await evaluate("selectFile("+str(index)+")")
    browser_results["selected_file"]=frows[index]["file_path"]
    browser_results["selected_declarations"]=await evaluate("document.querySelectorAll('#symbol-body tr').length")
    assert browser_results["selected_declarations"]==frows[index]["symbol_total"]
    browser_results["document_overflow"]=await evaluate("document.documentElement.scrollWidth>window.innerWidth")
    assert not browser_results["document_overflow"]
    shot=await cdp("Page.captureScreenshot",{"format":"png","captureBeyondViewport":True,"fromSurface":True})
    (OUT/"dashboard-screenshot.png").write_bytes(base64.b64decode(shot["data"]))
    browser_results.update(status="passed",browser="headless Chrome, new temporary profile",scripts_off_baseline="pre-rendered figures and 689 file rows")
    dump("browser-verification.json",browser_results)
    print(json.dumps(browser_results,indent=2))
finally:
    if ws is not None:ws.close()
    chrome_proc.terminate()
    try:chrome_proc.wait(timeout=5)
    except subprocess.TimeoutExpired:chrome_proc.kill();chrome_proc.wait()
    chrome_log.close()


{
  "initial_visible_files": 689,
  "unindexed_filter": 21,
  "test_file_filter": 175,
  "empty_search": 0,
  "keyboard_selection": "passed",
  "selected_file": "docs/reviews/2026-09-29-catalog-detach-fix/catalog-final-revision/catalog_tokens.before.rs",
  "selected_declarations": 18,
  "document_overflow": false,
  "status": "passed",
  "browser": "headless Chrome, new temporary profile",
  "scripts_off_baseline": "pre-rendered figures and 689 file rows"
}


In [30]:
print("Browser check reached",browser_results)
print("Durable audit kinds",[(r["id"],r["text"][:300]) for r in audits if "[[spur-audit v1]]" in r["text"] or "[[spur-signal v1]]" in r["text"]])


Browser check reached {'initial_visible_files': 689, 'unindexed_filter': 21, 'test_file_filter': 175, 'empty_search': 0}
Durable audit kinds [(2002, '[[spur-audit v1]]\n{"kind":"task-spec","task_id":"COV-AUDIT","context_files":[],"planned_write_files":["docs/research/artifacts/2026-11-02-rust-graph-coverage/audit/REPORT.md"],"profile":"codex","model":"gpt-6.1-sol","effort":"xhigh","agent":"codex"}'), (2005, '[[spur-audit v1]]\n{"kind":"dispatch","delegation_id":"44705f09fda54afc","worker":"codex","attempt":1}'), (2006, '[[spur-audit v1]]\n{"kind":"worker-started","delegation_id":"44705f09fda54afc","worker_branch":"spur/worker/v2/codex/99eaf8bce1ebb355/8f8879c9-39e4-4a4f-adaa-bb6b6a4fb0ef","worker_session_id":"8f8879c9-39e4-4a4f-adaa-bb6b6a4fb0ef","dispatched_base_oid":"edfb37be5d242814cdde6f4dd6e50d2856734038"}'), (2007, '[[spur-audit v1]]\n{"kind":"worker-evidence","audit_id":"d3fbba55-088f-44d2-b830-7721b8d09046","delegation_id":"44705f09fda54afc","brain_session_id":"99eaf8bce1ebb355

## Interpretation and audit limits

The published graph verifies **11,517 of 12,682** physical function/method declarations (**90.8%** across parsed files). All 11,517 in source-matching files are represented; the remaining callable declarations are in stale or unindexed files. Foreign declarations are a separate category.

The current file-local captures match **72,149 method sites** and **24,754 macro sites**, with **9,797** and **3,608** further sites unmeasured in six timeout files. These eligible populations differ from the published graph; the percentages are not a before/after improvement estimate.

Source classification uses file paths and inherited item annotation hints. `cfg_context` / `test_context` are **not exhaustive**: expression attributes and complex cfg formulas may be unmarked. A false flag does not imply an unconditional or production-only operation. Cargo configurations, expanded macros, target resolution, branch guards and panic completeness were not evaluated in this coverage run.

The independent audit is preserved under `audit/REPORT.md`. Direct inspection additionally checked every reported tuple-field / FFI correspondence gap against the captured node ranges. The bootstrap fragment's extra 115 method and 11 macro rows remain outside the full-module AST denominator.

In [46]:
(OUT/'inventory.evcxr.rs').write_text("use syn::spanned::Spanned;\nuse syn::visit::{self, Visit};\nuse proc_macro2::Span;\nuse serde_json::{json, Value};\n\nstruct RustInventory {\n    file: String, rows: Vec<Value>, test_context: bool, cfg_context: bool,\n}\nimpl RustInventory {\n    fn attr_flags(attrs: &[syn::Attribute]) -> (bool, bool) {\n        let test = attrs.iter().any(|a| a.path().segments.last().map(|s|s.ident == \"test\").unwrap_or(false)) ||\n            attrs.iter().any(|a| a.path().is_ident(\"cfg\") && a.parse_args::<syn::Path>().map(|p|p.is_ident(\"test\")).unwrap_or(false));\n        let cfg = attrs.iter().any(|a| a.path().is_ident(\"cfg\") || a.path().is_ident(\"cfg_attr\"));\n        (test, cfg)\n    }\n    fn push(&mut self, row_type: &str, kind: &str, name: String,\n            span: Span, ident: Option<Span>, body: Option<bool>) {\n        let range = span.byte_range();\n        let idrange = ident.map(|s| s.byte_range());\n        self.rows.push(json!({\n            \"file_path\": self.file, \"row_type\":row_type, \"kind\":kind, \"name\":name,\n            \"start\":range.start, \"core_start\":range.start, \"end\":range.end,\n            \"ident_start\":idrange.as_ref().map(|r|r.start),\n            \"ident_end\":idrange.as_ref().map(|r|r.end),\n            \"line_start\":span.start().line, \"line_end\":span.end().line,\n            \"test_context\":self.test_context, \"cfg_context\":self.cfg_context,\n            \"has_body\":body,\n        }));\n    }\n    fn named(&mut self, kind:&str, id:&syn::Ident, span:Span, body:Option<bool>) {\n        self.push(\"symbol\",kind,id.to_string(),span,Some(id.span()),body);\n    }\n}\nimpl<'ast> Visit<'ast> for RustInventory {\n    fn visit_item(&mut self, item:&'ast syn::Item) {\n        let attrs: &[syn::Attribute] = match item {\n            syn::Item::Const(x)=>&x.attrs, syn::Item::Enum(x)=>&x.attrs,\n            syn::Item::ExternCrate(x)=>&x.attrs, syn::Item::Fn(x)=>&x.attrs,\n            syn::Item::ForeignMod(x)=>&x.attrs, syn::Item::Impl(x)=>&x.attrs,\n            syn::Item::Macro(x)=>&x.attrs, syn::Item::Mod(x)=>&x.attrs,\n            syn::Item::Static(x)=>&x.attrs, syn::Item::Struct(x)=>&x.attrs,\n            syn::Item::Trait(x)=>&x.attrs, syn::Item::TraitAlias(x)=>&x.attrs,\n            syn::Item::Type(x)=>&x.attrs, syn::Item::Union(x)=>&x.attrs,\n            syn::Item::Use(x)=>&x.attrs, _=>&[],\n        };\n        let prev=(self.test_context,self.cfg_context);\n        let flags=Self::attr_flags(attrs);\n        self.test_context |= flags.0; self.cfg_context |= flags.1;\n        match item {\n            syn::Item::Fn(x)=>self.named(\"function\",&x.sig.ident,x.span(),Some(true)),\n            syn::Item::Struct(x)=>self.named(\"struct\",&x.ident,x.span(),None),\n            syn::Item::Enum(x)=>self.named(\"enum\",&x.ident,x.span(),None),\n            syn::Item::Union(x)=>self.named(\"union\",&x.ident,x.span(),None),\n            syn::Item::Trait(x)=>self.named(\"trait\",&x.ident,x.span(),None),\n            syn::Item::TraitAlias(x)=>self.named(\"trait_alias\",&x.ident,x.span(),None),\n            syn::Item::Type(x)=>self.named(\"type_alias\",&x.ident,x.span(),None),\n            syn::Item::Const(x)=>self.named(\"constant\",&x.ident,x.span(),None),\n            syn::Item::Static(x)=>self.named(\"static\",&x.ident,x.span(),None),\n            syn::Item::Mod(x)=>self.named(\"module\",&x.ident,x.span(),None),\n            syn::Item::Macro(x) if x.ident.is_some()=>self.named(\"macro_definition\",x.ident.as_ref().unwrap(),x.span(),None),\n            syn::Item::Impl(x)=>self.push(\"symbol\",\"impl\",\"<impl>\".into(),x.span(),None,None),\n            syn::Item::ForeignMod(x)=>self.push(\"symbol\",\"extern_block\",\"<extern>\".into(),x.span(),None,None),\n            _=>(),\n        }\n        visit::visit_item(self,item);\n        (self.test_context,self.cfg_context)=prev;\n    }\n    fn visit_impl_item(&mut self, item:&'ast syn::ImplItem) {\n        let prev=(self.test_context,self.cfg_context);\n        let attrs=match item {syn::ImplItem::Fn(x)=>&x.attrs, syn::ImplItem::Const(x)=>&x.attrs, syn::ImplItem::Type(x)=>&x.attrs, _=>&[][..]};\n        let flags=Self::attr_flags(attrs);self.test_context|=flags.0;self.cfg_context|=flags.1;\n        match item {\n            syn::ImplItem::Fn(x)=>self.named(\"method\",&x.sig.ident,x.span(),Some(true)),\n            syn::ImplItem::Const(x)=>self.named(\"associated_constant\",&x.ident,x.span(),None),\n            syn::ImplItem::Type(x)=>self.named(\"associated_type\",&x.ident,x.span(),None),\n            _=>(),\n        }\n        visit::visit_impl_item(self,item);\n        (self.test_context,self.cfg_context)=prev;\n    }\n    fn visit_trait_item(&mut self, item:&'ast syn::TraitItem) {\n        let prev=(self.test_context,self.cfg_context);\n        let attrs=match item {syn::TraitItem::Fn(x)=>&x.attrs, syn::TraitItem::Const(x)=>&x.attrs, syn::TraitItem::Type(x)=>&x.attrs, _=>&[][..]};\n        let flags=Self::attr_flags(attrs);self.test_context|=flags.0;self.cfg_context|=flags.1;\n        match item {\n            syn::TraitItem::Fn(x)=>self.named(\"trait_method\",&x.sig.ident,x.span(),Some(x.default.is_some())),\n            syn::TraitItem::Const(x)=>self.named(\"trait_constant\",&x.ident,x.span(),None),\n            syn::TraitItem::Type(x)=>self.named(\"trait_type\",&x.ident,x.span(),None),\n            _=>(),\n        }\n        visit::visit_trait_item(self,item);\n        (self.test_context,self.cfg_context)=prev;\n    }\n    fn visit_foreign_item(&mut self, item:&'ast syn::ForeignItem) {\n        match item {\n            syn::ForeignItem::Fn(x)=>self.named(\"foreign_function\",&x.sig.ident,x.span(),Some(false)),\n            syn::ForeignItem::Static(x)=>self.named(\"foreign_static\",&x.ident,x.span(),None),\n            syn::ForeignItem::Type(x)=>self.named(\"foreign_type\",&x.ident,x.span(),None),\n            _=>(),\n        }\n        visit::visit_foreign_item(self,item);\n    }\n    fn visit_variant(&mut self,x:&'ast syn::Variant) {\n        self.named(\"enum_variant\",&x.ident,x.span(),None);\n        visit::visit_variant(self,x);\n    }\n    fn visit_field(&mut self,x:&'ast syn::Field) {\n        if let Some(id)=&x.ident {self.named(\"field\",id,x.span(),None);}\n        else {self.push(\"symbol\",\"tuple_field\",\"<unnamed>\".into(),x.span(),None,None);}\n        visit::visit_field(self,x);\n    }\n    fn visit_expr(&mut self,x:&'ast syn::Expr) {\n        match x {\n            syn::Expr::Call(c)=>{\n                let name = match c.func.as_ref() {\n                    syn::Expr::Path(p)=>p.path.segments.last().map(|s|s.ident.to_string()).unwrap_or_default(),\n                    _=>\"<expression>\".into(),\n                };\n                self.push(\"callsite\",\"call\",name,c.span(),None,None);\n            },\n            syn::Expr::MethodCall(c)=>{\n                self.push(\"callsite\",\"method_call\",c.method.to_string(),c.span(),Some(c.method.span()),None);\n                self.rows.last_mut().unwrap()[\"core_start\"]=json!(c.receiver.span().byte_range().start);\n            },\n            syn::Expr::Index(c)=>self.push(\"operation\",\"index\",\"[]\".into(),c.span(),None,None),\n            syn::Expr::Closure(c)=>self.push(\"body\",\"closure\",\"<closure>\".into(),c.span(),None,Some(true)),\n            syn::Expr::Async(c)=>self.push(\"body\",\"async_block\",\"<async>\".into(),c.span(),None,Some(true)),\n            syn::Expr::Const(c)=>self.push(\"body\",\"const_block\",\"<const>\".into(),c.span(),None,Some(true)),\n            _=>(),\n        }\n        visit::visit_expr(self,x);\n    }\n    fn visit_macro(&mut self,x:&'ast syn::Macro) {\n        if !x.path.is_ident(\"macro_rules\") {\n            let name=x.path.segments.iter().map(|s|s.ident.to_string()).collect::<Vec<_>>().join(\"::\");\n            self.push(\"callsite\",\"macro_invocation\",name,x.span(),None,None);\n        }\n        // syn intentionally treats macro argument token trees as opaque.\n        visit::visit_macro(self,x);\n    }\n}\nfn inventory_rust(file:&str, source:&str)->Result<Vec<Value>,syn::Error>{\n    let ast=syn::parse_file(source)?;\n    let mut v=RustInventory{file:file.into(),rows:Vec::new(),test_context:false,cfg_context:false};\n    v.visit_file(&ast); Ok(v.rows)\n}\nlet calibration = \"// é 🦀\\n#[cfg(test)]\\npub fn r#loop() { fn nested() {} nested(); let é = 1; let f=||é; }\\nstruct S { field: u8 }\\nenum E { A(u8), B { x: bool } }\\ntrait T { fn required(&self); fn default(&self) {} }\\nimpl S { fn method(&self) {} }\\nmacro_rules! opaque { () => { fn generated() {} } }\\nfn probes() { let s = \\\"é\\\"; let _ = s.trim().len(); #[cfg(test)] s.trim(); assert!(true); }\\nunsafe extern \\\"C\\\" { fn foreign_probe(); }\\n\";\nlet cal=inventory_rust(\"calibration.rs\",calibration)?;\nassert_eq!(cal.iter().filter(|r|r[\"kind\"]==\"function\").count(),3);\nassert_eq!(cal.iter().filter(|r|r[\"kind\"]==\"trait_method\").count(),2);\nassert_eq!(cal.iter().filter(|r|r[\"kind\"]==\"macro_definition\").count(),1);\nassert!(!cal.iter().any(|r|r[\"name\"]==\"generated\"));\nfor r in cal.iter().filter(|r| !r[\"ident_start\"].is_null()) {\n    let start=r[\"ident_start\"].as_u64().unwrap() as usize;\n    let end=r[\"ident_end\"].as_u64().unwrap() as usize;\n    assert_eq!(&calibration[start..end],r[\"name\"].as_str().unwrap());\n}\nlet coverage_out=std::path::Path::new(\"/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-rust-graph-coverage\");\nstd::fs::write(coverage_out.join(\"calibration.rs\"),calibration)?;\nstd::fs::write(coverage_out.join(\"calibration-ast.json\"),serde_json::to_string_pretty(&cal)?)?;\nlet manifest:Value=serde_json::from_str(&std::fs::read_to_string(coverage_out.join(\"source-manifest.json\"))?)?;\nlet mut all=Vec::new(); let mut parses=Vec::new();\nfor file in manifest.as_array().unwrap() {\n    let path=file[\"path\"].as_str().unwrap();\n    let text=std::fs::read_to_string(coverage_out.join(\"source-snapshot\").join(path))?;\n    match inventory_rust(path,&text) {\n        Ok(rows)=>{parses.push(json!({\"file_path\":path,\"status\":\"parsed\",\"rows\":rows.len()}));all.extend(rows);}\n        Err(e)=>parses.push(json!({\"file_path\":path,\"status\":\"parse_error\",\"error\":e.to_string()})),\n    }\n}\nstd::fs::write(coverage_out.join(\"rust-ast.json\"),serde_json::to_string(&all)?)?;\nstd::fs::write(coverage_out.join(\"rust-parse-results.json\"),serde_json::to_string_pretty(&parses)?)?;\nprintln!(\"Independent physical AST rows: {}; parsed files: {}; parse errors: {}. Calibration passed (Unicode, raw identifiers, nested functions, trait declarations, opaque macros).\",\n    all.len(),parses.iter().filter(|r|r[\"status\"]==\"parsed\").count(),parses.iter().filter(|r|r[\"status\"]==\"parse_error\").count());\n")
(OUT/'inventory-dependencies.txt').write_text(':dep syn = { version = "=2.0.118", features = ["full", "visit", "extra-traits"] }\n:dep proc-macro2 = { version = "1", features = ["span-locations"] }\n:dep serde_json = "1"\n')
(OUT/'render-report.cell.py').write_text("# Self-contained scientific report: charts stay visible when scripts are disabled.\nfrom IPython.display import HTML, display\nimport html as html_lib\nbundle=spur.get(\"rust_coverage_bundle\")\nif hasattr(bundle,\"to_pandas\"): bundle=bundle.to_pandas()\nif isinstance(bundle,pd.DataFrame): bundle=bundle.iloc[0].to_dict()\nelif isinstance(bundle,list): bundle=bundle[0]\nassert isinstance(bundle,dict),type(bundle)\ns=json.loads(bundle[\"summary_json\"]);frows=json.loads(bundle[\"files_json\"]);srows=json.loads(bundle[\"symbols_json\"])\ndef h(x):return html_lib.escape(str(x),quote=True)\ndef svg_body(x):return x[x.index(\"<svg\"):x.rindex(\"</svg>\")+6]\ndef file_row(r,i):\n    return f'<tr data-index=\"{i}\" tabindex=\"0\"><td><button class=\"filelink\" data-file=\"{i}\">{h(r[\"file_path\"])}</button></td><td>{h(r[\"source_class\"])}</td><td>{h(r[\"published_file_status\"])}</td><td class=\"num\">{r[\"published_symbols\"]:,} / {r[\"symbol_total\"]:,}</td><td>{h(r[\"current_extraction_status\"])}</td><td class=\"num\">{r[\"current_symbols\"]:,}</td></tr>'\ntable=\"\".join(file_row(r,i) for i,r in enumerate(frows))\nlegend='<span><i style=\"background:#237a57\"></i>Source matches</span><span><i style=\"background:#bc7b16\"></i>Stale</span><span><i style=\"background:#ae414c\"></i>Unindexed</span>'\nheat=[];y=4\nfor area in sorted({r[\"area\"] for r in frows}):\n    ids=[i for i,r in enumerate(frows) if r[\"area\"]==area]\n    heat.append(f'<text x=\"0\" y=\"{y+14}\" font-size=\"12\" fill=\"#334155\">{h(area)} ({len(ids)})</text>')\n    for j,i in enumerate(ids):\n        r=frows[i];x=224+(j%36)*19;ty=y+(j//36)*19\n        color={\"source_match\":\"#237a57\",\"stale\":\"#bc7b16\",\"unindexed\":\"#ae414c\"}[r[\"published_file_status\"]]\n        heat.append(f'<rect x=\"{x}\" y=\"{ty}\" width=\"15\" height=\"15\" rx=\"2\" fill=\"{color}\" class=\"tile\" data-file=\"{i}\" tabindex=\"0\" role=\"button\" aria-label=\"{h(r[\"file_path\"])}\"><title>{h(r[\"file_path\"])} · {h(r[\"published_file_status\"])} · {r[\"published_symbols\"]}/{r[\"symbol_total\"]} declarations</title></rect>')\n    y+=max(1,(len(ids)+35)//36)*19+18\nheatmap=f'<svg id=\"filemap\" viewBox=\"0 0 920 {y}\" role=\"img\" aria-label=\"Each square is one captured Rust file\">{\"\".join(heat)}</svg>'\nclasses=\"\".join(f'<option value=\"{h(c)}\">{h(c.replace(\"_\",\" \"))}</option>' for c in sorted({r[\"source_class\"] for r in frows}))\npayload=json.dumps({\"files\":frows,\"symbols\":srows},ensure_ascii=True,separators=(\",\",\":\")).replace(\"<\",\"\\\\u003c\")\ncss=\"\"\"\n:root{color-scheme:light;font-family:system-ui,-apple-system,BlinkMacSystemFont,\"Segoe UI\",sans-serif;color:#233044;background:#f5f7fa}\n*{box-sizing:border-box}body{margin:0}main{max-width:1240px;margin:auto;padding:32px 28px 64px}\nh1{font-size:34px;letter-spacing:-.8px;margin:6px 0 10px}h2{font-size:20px;margin:0 0 12px}h3{font-size:16px;margin:16px 0 8px}\np{line-height:1.55;margin:8px 0 16px}.kicker{font-size:12px;letter-spacing:1.2px;color:#64748b;text-transform:uppercase}.sub{color:#526175;max-width:1000px}\n.metrics{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin:26px 0}.metric{padding:18px;border:1px solid #dce3eb;background:white;border-radius:6px}.metric strong{font-size:28px;display:block;font-variant-numeric:tabular-nums}.metric span{font-size:12px;color:#526175}.metric b{font-size:13px;display:block;margin:5px 0;font-weight:600}\nsection{margin:24px 0;background:white;border:1px solid #dce3eb;border-radius:6px;padding:24px}\nsvg{max-width:100%;height:auto}.figure svg{display:block;width:100%}.legend{display:flex;gap:18px;flex-wrap:wrap;font-size:12px;color:#526175;margin:12px 0}.legend span{display:flex;align-items:center;gap:6px}.legend i{width:12px;height:12px;border-radius:2px;display:inline-block}\n.tile{cursor:pointer}.tile:hover,.tile:focus{stroke:#0f172a;stroke-width:3;outline:none}\n.controls{display:flex;gap:12px;align-items:end;flex-wrap:wrap;margin:18px 0}label{font-size:12px;display:flex;gap:5px;flex-direction:column;color:#475569}input,select,button{font:inherit}input,select{padding:9px 10px;border:1px solid #b9c5d3;border-radius:4px;background:white;color:#233044}\ninput{width:290px}button{cursor:pointer}button.filelink{border:0;padding:0;background:none;color:#214f7c;font:inherit;text-align:left;overflow-wrap:anywhere}button.filelink:hover{text-decoration:underline}\n.scroll{max-height:440px;overflow:auto;border:1px solid #e2e8f0}table{border-collapse:collapse;width:100%;font-size:12px;text-align:left}th{position:sticky;top:0;background:#f1f5f9;z-index:1;white-space:nowrap;font-weight:600}td,th{padding:9px 10px;border-bottom:1px solid #e5ebf1;vertical-align:top}tr:hover{background:#f4f8fc}.num{font-family:ui-monospace,monospace;text-align:right;white-space:nowrap}code{font-family:ui-monospace,monospace;font-size:.9em;overflow-wrap:anywhere}.note{font-size:12px;color:#526175}.detail{background:#f7f9fc;padding:16px;border:1px solid #e2e8f0;border-radius:4px;margin:16px 0}.detail h3{overflow-wrap:anywhere}.detail table{background:white}.detail .scroll{max-height:300px}details summary{cursor:pointer;font-weight:600}\n#rowcount{font-size:12px;color:#64748b}#selected-status{font-size:13px}#symbol-strip{width:100%;height:94px}\nfooter{font-size:12px;color:#64748b;line-height:1.7}\n@media(max-width:800px){main{padding:20px 14px}.metrics{grid-template-columns:repeat(2,1fr)}section{padding:16px}h1{font-size:28px}.scroll{max-height:360px}}\n@media print{body{background:white}main{padding:0}.controls{display:none}.scroll{max-height:none;overflow:visible}.metric,section{break-inside:avoid}}\n\"\"\"\njs=r\"\"\"\nconst data=JSON.parse(document.getElementById(\"coverage-data\").textContent);\nconst esc=x=>String(x).replace(/[&<>\"']/g,c=>({\"&\":\"&amp;\",\"<\":\"&lt;\",\">\":\"&gt;\",\"\\\"\":\"&quot;\",\"'\":\"&#39;\"}[c]));\nconst colors={represented:\"#237a57\",stale:\"#bc7b16\",unindexed:\"#ae414c\",no_matching_symbol:\"#5544a8\",timeout:\"#9babc0\",ambiguous:\"#a855f7\"};\nconst symbolMap=new Map();\nfor(const row of data.symbols){if(!symbolMap.has(row[0]))symbolMap.set(row[0],[]);symbolMap.get(row[0]).push(row);}\nfunction selectFile(i){\n const f=data.files[i], syms=symbolMap.get(f.file_path)||[];\n document.getElementById(\"selected-name\").textContent=f.file_path;\n document.getElementById(\"selected-status\").textContent=f.lines.toLocaleString()+\" physical lines · Published: \"+f.published_file_status+\" (\"+f.published_symbols+\"/\"+f.symbol_total+\" declaration nodes verified) · Current extraction: \"+f.current_extraction_status+\" (\"+f.current_symbols+\"/\"+f.symbol_total+\" represented) · Parser: \"+f.parser_status;\n document.getElementById(\"selected-hash\").textContent=\"Source SHA256 \"+f.sha256;\n const denom=Math.max(f.lines,1);let drawing='<text x=\"0\" y=\"12\" font-size=\"11\" fill=\"#526175\">Declaration spans along physical file lines 1–'+f.lines+'. Containers overlap; this is not line execution coverage.</text>';\n for(const [statusIndex,y,label] of [[5,24,\"Published\"],[6,56,\"Current\"]]){\n  drawing+='<text x=\"0\" y=\"'+(y+11)+'\" font-size=\"11\" fill=\"#526175\">'+label+'</text><rect x=\"78\" y=\"'+y+'\" width=\"900\" height=\"16\" fill=\"#e8edf3\"/>';\n  const ordered=[...syms].sort((a,b)=>(a[statusIndex]===\"represented\"?-1:1)-(b[statusIndex]===\"represented\"?-1:1));\n  for(const r of ordered){\n   const x=78+900*(r[3]-1)/denom,w=Math.max(1,900*(r[4]-r[3]+1)/denom);\n   drawing+='<rect x=\"'+x+'\" y=\"'+y+'\" width=\"'+w+'\" height=\"16\" fill=\"'+(colors[r[statusIndex]]||\"#64748b\")+'\" opacity=\".8\"><title>'+esc(r[1])+\" \"+esc(r[2])+\" · lines \"+r[3]+\"–\"+r[4]+\" · \"+esc(r[statusIndex])+\"</title></rect>\";\n  }\n }\n document.getElementById(\"symbol-strip\").innerHTML=drawing;\n document.getElementById(\"symbol-body\").innerHTML=syms.map(r=>\"<tr><td>\"+esc(r[1])+\"</td><td><code>\"+esc(r[2])+'</code></td><td class=\"num\">'+r[3]+\"–\"+r[4]+\"</td><td>\"+esc(r[5])+\"</td><td>\"+esc(r[6])+\"</td></tr>\").join(\"\");\n}\nfunction filterFiles(){\n const q=document.getElementById(\"search\").value.toLowerCase(),cls=document.getElementById(\"class-filter\").value,state=document.getElementById(\"state-filter\").value;let n=0;\n for(const row of document.querySelectorAll(\"#file-body tr\")){\n  const f=data.files[Number(row.dataset.index)],ok=(!q||f.file_path.toLowerCase().includes(q))&&(!cls||f.source_class===cls)&&(!state||f.published_file_status===state);\n  row.hidden=!ok;if(ok)n++;\n }\n for(const tile of document.querySelectorAll(\".tile\")){\n  const f=data.files[Number(tile.dataset.file)],ok=(!q||f.file_path.toLowerCase().includes(q))&&(!cls||f.source_class===cls)&&(!state||f.published_file_status===state);\n  tile.style.opacity=ok?\"1\":\".12\";\n }\n document.getElementById(\"rowcount\").textContent=n+\" / \"+data.files.length+\" captured Rust files shown\";\n}\ndocument.addEventListener(\"click\",e=>{const el=e.target.closest(\"[data-file]\");if(el)selectFile(Number(el.dataset.file));});\ndocument.getElementById(\"filemap\").addEventListener(\"keydown\",e=>{if((e.key===\"Enter\"||e.key===\" \")&&e.target.dataset.file!==undefined){e.preventDefault();selectFile(Number(e.target.dataset.file));}});\nfor(const id of [\"search\",\"class-filter\",\"state-filter\"])document.getElementById(id).addEventListener(\"input\",filterFiles);\nfilterFiles();selectFile(Math.max(0,data.files.findIndex(f=>f.published_file_status===\"unindexed\"&&f.symbol_total>100)));\n\"\"\"\nsite_count=lambda kind,status:sum(r[\"count\"] for r in s[\"site_states\"] if r[\"kind\"]==kind and r[\"current_status\"]==status)\nmethod_count=site_count(\"method_call\",\"represented\");method_unknown=site_count(\"method_call\",\"timeout\")\nmacro_count=site_count(\"macro_invocation\",\"represented\");macro_unknown=site_count(\"macro_invocation\",\"timeout\")\nhtml_doc=f\"\"\"<!doctype html><html lang=\"en\"><head><meta charset=\"utf-8\"><meta name=\"viewport\" content=\"width=device-width,initial-scale=1\"><title>Rust source and Spur graph coverage</title><style>{css}</style></head><body><main>\n<header><div class=\"kicker\">Measured source comparison · 02 November 2026</div><h1>Rust source and Spur graph coverage</h1><p class=\"sub\">Physical Rust declarations and invocation sites in <code>spur-notebook</code>, independently parsed with <code>syn 2.0.118</code>. The published graph is source-hash gated; current-extractor results come from separate file-local captures.</p></header>\n<div class=\"metrics\">\n<div class=\"metric\"><strong>{s[\"files\"]:,}</strong><b>captured Rust files</b><span>{s[\"parsed_files\"]:,} complete modules parsed; {s[\"parse_errors\"]} include fragment excluded</span></div>\n<div class=\"metric\"><strong>{s[\"published_callable_states\"][\"represented\"]:,} / {s[\"callables\"]:,}</strong><b>callable declarations verified in published graph</b><span>{s[\"published_callable_states\"][\"represented\"]/s[\"callables\"]:.1%} of parsed physical declarations; {s[\"published_callable_states\"].get(\"stale\",0):,} stale, {s[\"published_callable_states\"].get(\"unindexed\",0):,} unindexed</span></div>\n<div class=\"metric\"><strong>{method_count:,}</strong><b>method sites matched in current captures</b><span>{method_unknown:,} further sites unmeasured in {s[\"current_extraction_states\"].get(\"timeout\",0)} timed-out files</span></div>\n<div class=\"metric\"><strong>{macro_count:,}</strong><b>macro sites matched in current captures</b><span>{macro_unknown:,} further sites unmeasured; expansion remains opaque</span></div>\n</div>\n<section class=\"figure\"><h2>Coverage overview</h2>{svg_body(bundle[\"overview_svg\"])}</section>\n<section><h2>Each square is one physical Rust file</h2><p class=\"note\">Color shows published graph freshness. Click a square or filename to inspect declaration kinds and source line spans. Tests, examples, vendored code and research copies remain separate filters.</p><div class=\"legend\">{legend}</div>{heatmap}\n<div class=\"controls\"><label>Find a file<input id=\"search\" type=\"search\" placeholder=\"e.g. telemetry or state.rs\" autocomplete=\"off\"></label><label>Source category<select id=\"class-filter\"><option value=\"\">All categories</option>{classes}</select></label><label>Published graph<select id=\"state-filter\"><option value=\"\">All source states</option><option value=\"source_match\">Source matches</option><option value=\"stale\">Stale</option><option value=\"unindexed\">Unindexed</option></select></label><span id=\"rowcount\" aria-live=\"polite\">{len(frows)} captured files</span></div>\n<div id=\"file-detail\" class=\"detail\"><h3 id=\"selected-name\">Select a file to inspect its declaration spans</h3><p id=\"selected-status\">The file table below remains available when scripts are disabled.</p><code id=\"selected-hash\"></code><svg id=\"symbol-strip\" viewBox=\"0 0 1000 94\" aria-label=\"Declaration spans in selected source file\"></svg><div class=\"scroll\"><table><thead><tr><th>Kind</th><th>Name</th><th>Source lines</th><th>Published</th><th>Current extraction</th></tr></thead><tbody id=\"symbol-body\"></tbody></table></div></div>\n<div class=\"scroll\"><table><thead><tr><th>Rust file</th><th>Source category</th><th>Published file</th><th>Verified / declarations</th><th>Current extraction</th><th>Current declarations</th></tr></thead><tbody id=\"file-body\">{table}</tbody></table></div></section>\n<section class=\"figure\"><details><summary>Declaration coverage by kind</summary>{svg_body(bundle[\"kinds_svg\"])}</details></section>\n<section><h2>Where the evidence stops</h2><p>On matching source files, every parsed function, method and trait-method declaration has a symbol node. Unnamed tuple fields and foreign declarations lack individual nodes; their parent text can still be present. The current capture exposes method and macro sites, while free-call invocation identity is not measured by this capture API.</p><p>{s[\"current_extraction_states\"].get(\"timeout\",0)} current-extractor files exceeded a 30-second budget. They account for {s[\"current_declaration_states\"].get(\"timeout\",0):,} declarations, {method_unknown:,} method sites and {macro_unknown:,} macro sites; gray means unmeasured. One incomplete bootstrap fragment has no full-module AST denominator.</p><p>Macro-generated code, Cargo feature selection, trait dispatch, implicit compiler checks, branch feasibility and runtime reachability need separate evidence. This report does not establish absence of panics.</p></section>\n<footer>Published schema: <code>{h(s[\"published_schema\"])}</code> · Graph hash: <code>{h(s[\"graph_hash\"])}</code><br>All symbol matches require source identity, declaration kind, name, identifier containment and exact declaration end. Attribute-bearing method expressions normalize to their receiver start and retain exact method-token/end positions.<br>Generated-source markers are a comment heuristic; no marker does not prove handwritten origin. Physical line totals include comments and whitespace.</footer>\n<script id=\"coverage-data\" type=\"application/json\">{payload}</script><script>{js}</script></main></body></html>\"\"\"\n(OUT/\"coverage.html\").write_text(html_doc)\n(OUT/\"dashboard.js\").write_text(js)\nassert len(frows)==s[\"files\"] and len(srows)==s[\"declarations\"]\ndisplay(HTML(html_doc))\n")
(OUT/'render-charts.cell.py').write_text("import matplotlib\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nfrom matplotlib.ticker import PercentFormatter\nimport io\nfiles_df=pd.read_parquet(OUT/\"file-coverage.parquet\")\nsym_df=pd.read_parquet(OUT/\"symbol-coverage.parquet\")\ncall_df=pd.read_parquet(OUT/\"callsite-coverage.parquet\")\nsummary=json.loads((OUT/\"summary.json\").read_text())\nCOLORS={\"represented\":\"#237a57\",\"source_match\":\"#237a57\",\"stale\":\"#bc7b16\",\"unindexed\":\"#ae414c\",\"no_matching_symbol\":\"#5544a8\",\"timeout\":\"#9babc0\",\"free_call_site_not_exposed_by_capture\":\"#64748b\"}\nplt.rcParams.update({\"font.family\":\"DejaVu Sans\",\"font.size\":10,\"axes.spines.top\":False,\"axes.spines.right\":False,\"svg.fonttype\":\"none\",\"figure.facecolor\":\"white\",\"axes.labelcolor\":\"#233044\",\"text.color\":\"#233044\",\"xtick.color\":\"#475569\",\"ytick.color\":\"#475569\"})\nfig,axes=plt.subplots(2,2,figsize=(16,11.5),gridspec_kw={\"height_ratios\":[1,1.35]})\nfig.suptitle(\"Rust syntax represented by Spur graph · spur-notebook\",x=.02,ha=\"left\",y=.99,fontsize=21,fontweight=\"bold\")\nfig.text(.02,.946,\"689 captured Rust files · independent syn AST · published v10 versus current file-local extractor\",fontsize=11,color=\"#526175\")\ndef stack(ax,frame,column,order,title):\n    grouped=frame.groupby(column).size()\n    left=0\n    for status in order:\n        n=int(grouped.get(status,0))\n        if not n:continue\n        ax.barh([0],[n],left=left,color=COLORS[status],height=.5,label=f'{status.replace(\"_\",\" \")} · {n:,}')\n        if n/len(frame)>.06:ax.text(left+n/2,0,f\"{n:,}\\n{n/len(frame):.1%}\",ha=\"center\",va=\"center\",color=\"white\",fontweight=\"bold\")\n        left+=n\n    ax.set_xlim(0,len(frame));ax.set_yticks([]);ax.set_xlabel(f\"Out of {len(frame):,} \"+(\"Rust files\" if column==\"published_file_status\" else \"callable declarations\"))\n    ax.set_title(title,loc=\"left\",pad=14,fontweight=\"bold\")\n    ax.legend(loc=\"upper left\",bbox_to_anchor=(0,-.28),ncol=2,frameon=False,fontsize=9)\nstack(axes[0,0],files_df,\"published_file_status\",[\"source_match\",\"stale\",\"unindexed\"],\"Published graph: file freshness\")\ncallables=sym_df[sym_df.kind.isin([\"function\",\"method\",\"trait_method\"])]\nstack(axes[0,1],callables,\"published_status\",[\"represented\",\"stale\",\"unindexed\"],\"Published graph: function / method declarations\")\narea=sym_df.groupby([\"area\",\"published_status\"]).size().unstack(fill_value=0)\narea=area.loc[area.sum(axis=1).sort_values(ascending=True).index]\nleft=[0.0]*len(area)\nfor status in [\"represented\",\"stale\",\"unindexed\",\"no_matching_symbol\"]:\n    vals=area.get(status,pd.Series(0,index=area.index))/area.sum(axis=1)\n    axes[1,0].barh(area.index,vals,left=left,label=status.replace(\"_\",\" \"),color=COLORS[status])\n    left=[a+b for a,b in zip(left,vals)]\naxes[1,0].xaxis.set_major_formatter(PercentFormatter(1));axes[1,0].set_xlim(0,1)\naxes[1,0].set_title(\"Published graph: declaration coverage by source area\",loc=\"left\",fontweight=\"bold\",pad=14)\naxes[1,0].set_xlabel(\"Share of parsed declarations; includes fields and containers\")\naxes[1,0].legend(loc=\"upper left\",bbox_to_anchor=(0,-.18),ncol=2,frameon=False,fontsize=9)\nsite=call_df[call_df.kind.isin([\"method_call\",\"macro_invocation\"])]\nsite=site.groupby([\"kind\",\"current_status\"]).size().unstack(fill_value=0)\nleft=[0]*len(site)\nfor status in [\"represented\",\"timeout\"]:\n    vals=site.get(status,pd.Series(0,index=site.index)).tolist()\n    axes[1,1].barh([\"Macro invocations\" if k==\"macro_invocation\" else \"Method invocations\" for k in site.index],vals,left=left,color=COLORS[status],label=\"exact site match\" if status==\"represented\" else \"30s timeout: unmeasured\")\n    for i,(start,v) in enumerate(zip(left,vals)):\n        if v:\n            narrow=v<max(left+vals)*.12\n            axes[1,1].text(start+v+800 if narrow else start+v/2,i,f\"{v:,}\",ha=\"left\" if narrow else \"center\",va=\"center\",color=\"#475569\" if narrow else \"white\",fontweight=\"bold\")\n    left=[a+b for a,b in zip(left,vals)]\naxes[1,1].set_xlim(0,max(left)*1.06)\naxes[1,1].set_title(\"Current extractor: exact source site identities\",loc=\"left\",fontweight=\"bold\",pad=14)\naxes[1,1].set_xlabel(\"Physical AST invocations; macro expansion stays opaque\")\naxes[1,1].legend(loc=\"upper left\",bbox_to_anchor=(0,-.18),frameon=False,fontsize=9)\nfig.text(.02,.018,\"Scope: 688 complete Rust files parsed; one include fragment excluded from AST denominators. Six current-extractor files timed out.\\nRepresentation does not establish call-target correctness, branch feasibility, execution coverage, or absence of panics.\",fontsize=10,color=\"#526175\")\nfig.subplots_adjust(top=.88,bottom=.17,hspace=.87,wspace=.53,left=.19,right=.98)\nfig.savefig(OUT/\"coverage-overview.svg\",bbox_inches=\"tight\")\nfig.savefig(OUT/\"coverage-overview.png\",dpi=150,bbox_inches=\"tight\")\nplt.close(fig)\n# Detailed declaration-kind comparison uses the same denominator in both panels.\nkinds=sym_df.groupby(\"kind\").size().sort_values().index\nfig,axs=plt.subplots(1,2,figsize=(15,10),sharey=True)\nfor ax,column,title,statuses in [\n    (axs[0],\"published_status\",\"Published graph · source-hash gated\",[\"represented\",\"stale\",\"unindexed\",\"no_matching_symbol\"]),\n    (axs[1],\"current_status\",\"Current extractor · file-local\",[\"represented\",\"timeout\",\"no_matching_symbol\"])]:\n    data=sym_df.groupby([\"kind\",column]).size().unstack(fill_value=0).reindex(kinds)\n    left=[0.0]*len(data)\n    for status in statuses:\n        vals=data.get(status,pd.Series(0,index=kinds))/data.sum(axis=1)\n        ax.barh([k.replace(\"_\",\" \") for k in kinds],vals,left=left,color=COLORS[status],label=status.replace(\"_\",\" \"))\n        left=[a+b for a,b in zip(left,vals)]\n    ax.set_xlim(0,1);ax.xaxis.set_major_formatter(PercentFormatter(1))\n    ax.set_title(title,loc=\"left\",fontweight=\"bold\");ax.legend(loc=\"upper left\",bbox_to_anchor=(0,-.07),frameon=False,fontsize=9,ncol=2)\naxs[0].set_ylabel(\"Physical AST declaration kind\")\nfig.suptitle(\"Where declaration coverage ends\",fontsize=20,x=.02,ha=\"left\",fontweight=\"bold\")\nfig.text(.02,.005,\"Tuple fields are present inside their parent type's bytes, but lack individual symbol nodes. Foreign declarations lack symbol nodes.\\nA gray segment means extraction exceeded its time budget, not that the source syntax is unsupported.\",fontsize=10,color=\"#526175\")\nfig.subplots_adjust(left=.18,top=.9,bottom=.16,wspace=.12)\nfig.savefig(OUT/\"coverage-by-kind.svg\",bbox_inches=\"tight\")\nfig.savefig(OUT/\"coverage-by-kind.png\",dpi=150,bbox_inches=\"tight\")\nplt.close(fig)\nfor svg_name in [\"coverage-overview.svg\",\"coverage-by-kind.svg\"]:\n    svg_path=OUT/svg_name\n    svg_path.write_text(\"\\n\".join(line.rstrip() for line in svg_path.read_text().splitlines())+\"\\n\")\ncompact_symbols=sym_df[[\"file_path\",\"kind\",\"name\",\"line_start\",\"line_end\",\"published_status\",\"current_status\"]].values.tolist()\npayload={\"symbols_json\":json.dumps(compact_symbols),\"summary_json\":json.dumps(summary),\"files_json\":files_df.to_json(orient=\"records\"),\"overview_svg\":(OUT/\"coverage-overview.svg\").read_text(),\"kinds_svg\":(OUT/\"coverage-by-kind.svg\").read_text()}\nspur.put(\"rust_coverage_bundle\",pd.DataFrame([payload]))\nprint(\"Exported SVG and PNG overview / kind charts; published rust_coverage_bundle (one Arrow port).\")\n")
(OUT/'browser-check.cell.txt').write_text("# Offline browser smoke check in a fresh profile, with no user browser state.\nimport asyncio, urllib.request, base64, uuid\nfrom tornado.websocket import websocket_connect\nCHROME=\"/Applications/Google Chrome.app/Contents/MacOS/Google Chrome\"\nPROFILE=PRIVATE/(\"chrome-profile-\"+str(uuid.uuid4()))\nchrome_log=open(OUT/\"browser.log\",\"wb\")\nchrome_proc=subprocess.Popen([CHROME,\"--headless=new\",\"--remote-debugging-port=0\",\"--user-data-dir=\"+str(PROFILE),\"--disable-background-networking\",\"--no-first-run\",\"--no-default-browser-check\",\"--disable-extensions\",\"about:blank\"],stdout=chrome_log,stderr=chrome_log)\nbrowser_results={};ws=None\ntry:\n    for attempt in range(100):\n        if (PROFILE/\"DevToolsActivePort\").exists():break\n        if chrome_proc.poll() is not None:raise RuntimeError(\"Chrome exited; inspect browser.log\")\n        await asyncio.sleep(.1)\n    port=int((PROFILE/\"DevToolsActivePort\").read_text().splitlines()[0])\n    request=urllib.request.Request(\"http://127.0.0.1:\"+str(port)+\"/json/new?about:blank\",method=\"PUT\")\n    tab=json.load(urllib.request.urlopen(request))\n    ws=await websocket_connect(tab[\"webSocketDebuggerUrl\"])\n    cdp_id=0\n    async def cdp(method,params=None):\n        global cdp_id\n        cdp_id+=1;cid=cdp_id\n        await ws.write_message(json.dumps({\"id\":cid,\"method\":method,\"params\":params or {}}))\n        while True:\n            message=json.loads(await asyncio.wait_for(ws.read_message(),10))\n            if message.get(\"id\")==cid:\n                assert \"error\" not in message,message\n                return message.get(\"result\",{})\n    async def evaluate(code):\n        result=await cdp(\"Runtime.evaluate\",{\"expression\":code,\"returnByValue\":True,\"awaitPromise\":True})\n        assert \"exceptionDetails\" not in result,result\n        return result[\"result\"].get(\"value\")\n    await cdp(\"Page.enable\");await cdp(\"Runtime.enable\")\n    await cdp(\"Emulation.setDeviceMetricsOverride\",{\"width\":1440,\"height\":1100,\"deviceScaleFactor\":1,\"mobile\":False})\n    await cdp(\"Page.navigate\",{\"url\":(OUT/\"coverage.html\").as_uri()})\n    for attempt in range(100):\n        if await evaluate(\"document.readyState==='complete' && typeof selectFile==='function'\"):break\n        await asyncio.sleep(.1)\n    assert await evaluate(\"typeof selectFile==='function'\")\n    browser_results[\"initial_visible_files\"]=await evaluate(\"document.querySelectorAll('#file-body tr:not([hidden])').length\")\n    assert browser_results[\"initial_visible_files\"]==689\n    await evaluate(\"document.getElementById('state-filter').value='unindexed';filterFiles()\")\n    browser_results[\"unindexed_filter\"]=await evaluate(\"document.querySelectorAll('#file-body tr:not([hidden])').length\")\n    assert browser_results[\"unindexed_filter\"]==21\n    await evaluate(\"document.getElementById('state-filter').value='';document.getElementById('class-filter').value='test_file';filterFiles()\")\n    browser_results[\"test_file_filter\"]=await evaluate(\"document.querySelectorAll('#file-body tr:not([hidden])').length\")\n    assert browser_results[\"test_file_filter\"]==sum(r[\"source_class\"]==\"test_file\" for r in frows)\n    await evaluate(\"document.getElementById('class-filter').value='';document.getElementById('search').value='__no_such_rust_source__';filterFiles()\")\n    browser_results[\"empty_search\"]=await evaluate(\"document.querySelectorAll('#file-body tr:not([hidden])').length\")\n    assert browser_results[\"empty_search\"]==0\n    await evaluate(\"document.getElementById('search').value='';filterFiles();document.querySelector('.tile').dispatchEvent(new KeyboardEvent('keydown',{key:'Enter',bubbles:true}))\")\n    first_tile_index=await evaluate(\"Number(document.querySelector('.tile').dataset.file)\")\n    assert await evaluate(\"document.getElementById('selected-name').textContent\")==frows[first_tile_index][\"file_path\"]\n    browser_results[\"keyboard_selection\"]=\"passed\"\n    index=next(i for i,r in enumerate(frows) if r[\"current_gaps\"]>0 and r[\"published_file_status\"]==\"source_match\")\n    await evaluate(\"selectFile(\"+str(index)+\")\")\n    browser_results[\"selected_file\"]=frows[index][\"file_path\"]\n    browser_results[\"selected_declarations\"]=await evaluate(\"document.querySelectorAll('#symbol-body tr').length\")\n    assert browser_results[\"selected_declarations\"]==frows[index][\"symbol_total\"]\n    browser_results[\"document_overflow\"]=await evaluate(\"document.documentElement.scrollWidth>window.innerWidth\")\n    assert not browser_results[\"document_overflow\"]\n    shot=await cdp(\"Page.captureScreenshot\",{\"format\":\"png\",\"captureBeyondViewport\":True,\"fromSurface\":True})\n    (OUT/\"dashboard-screenshot.png\").write_bytes(base64.b64decode(shot[\"data\"]))\n    browser_results.update(status=\"passed\",browser=\"headless Chrome, new temporary profile\",scripts_off_baseline=\"pre-rendered figures and 689 file rows\")\n    dump(\"browser-verification.json\",browser_results)\n    print(json.dumps(browser_results,indent=2))\nfinally:\n    if ws is not None:ws.close()\n    chrome_proc.terminate()\n    try:chrome_proc.wait(timeout=5)\n    except subprocess.TimeoutExpired:chrome_proc.kill();chrome_proc.wait()\n    chrome_log.close()\n")
(OUT/'audit').mkdir(exist_ok=True)
report_bytes=subprocess.check_output(['git','-C','/Users/kevintruong/project/SpurCode','show','2e4a8580afd0d53dded1a383f5c7311a363a3f8c:docs/research/artifacts/2026-11-02-rust-graph-coverage/audit/REPORT.md'])
(OUT/'audit/REPORT.md').write_bytes(report_bytes)
print('Saved exact experiment cell sources and independent audit; cell scripts expect notebook kernel state.')


Saved exact experiment cell sources and independent audit; cell scripts expect notebook kernel state.


In [44]:
(OUT/'verify.py').write_text("\"\"\"Verify captured Rust coverage without building either project.\nWorks with loose inputs or the bundled evidence.tar.gz; never parses a notebook.\n\"\"\"\nfrom pathlib import Path, PurePosixPath\nfrom collections import Counter\nfrom html.parser import HTMLParser\nimport json, gzip, hashlib, io, tarfile\nimport pandas as pd\n\nclass Inputs:\n    def __init__(self,root):self.root=Path(root);self.archive=None\n    def read(self,name):\n        p=self.root/name\n        if p.is_file():return p.read_bytes()\n        if self.archive is None:\n            with tarfile.open(self.root/\"evidence.tar.gz\",\"r:gz\") as t:\n                self.archive={}\n                for member in t:\n                    assert member.isfile() and not PurePosixPath(member.name).is_absolute() and \"..\" not in PurePosixPath(member.name).parts\n                    self.archive[member.name]=t.extractfile(member).read()\n        return self.archive[name]\n    def js(self,name):return json.loads(self.read(name))\n    def frame(self,name):return pd.read_parquet(io.BytesIO(self.read(name)))\n\nclass Page(HTMLParser):\n    def __init__(self):super().__init__();self.file_rows=0;self.tile_count=0;self.payload=False;self.parts=[];self.external=[]\n    def handle_starttag(self,tag,attrs):\n        a=dict(attrs)\n        if tag==\"tr\" and \"data-index\" in a:self.file_rows+=1\n        if tag==\"rect\" and a.get(\"class\")==\"tile\":self.tile_count+=1\n        if tag==\"script\" and a.get(\"id\")==\"coverage-data\":self.payload=True\n        for key in (\"src\",\"href\",\"xlink:href\"):\n            if a.get(key,\"\").startswith((\"http://\",\"https://\",\"//\")):self.external.append((tag,key,a[key]))\n    def handle_endtag(self,tag):\n        if tag==\"script\":self.payload=False\n    def handle_data(self,data):\n        if self.payload:self.parts.append(data)\n\ndef verify(root):\n    inp=Inputs(root);manifest=inp.js(\"source-manifest.json\");summary=inp.js(\"summary.json\")\n    paths={r[\"path\"]:r for r in manifest};assert len(paths)==len(manifest)==summary[\"files\"]\n    source={}\n    for p,r in paths.items():\n        b=inp.read(\"source-snapshot/\"+p);source[p]=b\n        assert len(b)==r[\"bytes\"] and len(b.splitlines())==r[\"lines\"]\n        assert hashlib.sha256(b).hexdigest()==r[\"sha256\"]\n        assert hashlib.sha1(b\"blob \"+str(len(b)).encode()+b\"\\0\"+b).hexdigest()==r[\"git_blob_oid\"]\n    assert sum(len(b) for b in source.values())==summary[\"source_bytes\"]\n    ast=inp.js(\"rust-ast.json\");parses=inp.js(\"rust-parse-results.json\")\n    assert len(parses)==len(paths) and set(r[\"file_path\"] for r in parses)==set(paths)\n    assert Counter(r[\"file_path\"] for r in ast)==Counter({r[\"file_path\"]:r[\"rows\"] for r in parses if r[\"status\"]==\"parsed\" and r[\"rows\"]})\n    for r in ast:\n        assert 0<=r[\"start\"]<=r.get(\"core_start\",r[\"start\"])<=r[\"end\"]<=len(source[r[\"file_path\"]])\n        if r[\"ident_start\"] is not None:\n            assert source[r[\"file_path\"]][r[\"ident_start\"]:r[\"ident_end\"]].decode()==r[\"name\"]\n    syms=inp.frame(\"symbol-coverage.parquet\");calls=inp.frame(\"callsite-coverage.parquet\");files=inp.frame(\"file-coverage.parquet\")\n    assert len(syms)==sum(r[\"row_type\"]==\"symbol\" for r in ast)==summary[\"declarations\"]\n    assert len(calls)==sum(r[\"row_type\"]==\"callsite\" for r in ast)\n    fm={r[\"path\"]:r[\"content_oid\"] for r in inp.frame(\"graph-file_manifests.parquet\").to_dict(\"records\")}\n    pub=inp.frame(\"graph-nodes.parquet\").set_index(\"stable_symbol_id\").to_dict(\"index\")\n    recs=inp.js(\"current-extractor-results.json\");assert len(recs)==len(paths)\n    current={};sites={};raw_site_counts=Counter();external_repeats=0\n    for rec in recs:\n        path=rec[\"path\"];assert rec[\"source_sha256\"]==paths[path][\"sha256\"]\n        if rec[\"status\"]!=\"success\":\n            assert rec[\"status\"]==\"timeout\"\n            continue\n        raw=gzip.decompress(inp.read(\"current-extractor/\"+rec[\"artifact\"]))\n        assert hashlib.sha256(raw).hexdigest()==rec[\"stdout_sha256\"]\n        graph=json.loads(raw);gfm=graph[\"graph\"][\"file_manifests\"]\n        assert len(gfm)==1 and gfm[0][\"path\"]==path and gfm[0][\"content_oid\"]==paths[path][\"git_blob_oid\"]\n        for n in graph[\"graph\"][\"symbols\"]:\n            if n[\"stable_symbol_id\"] in current:\n                assert n[\"symbol_kind\"]==\"external\" and current[n[\"stable_symbol_id\"]]==n\n                external_repeats+=1\n            current[n[\"stable_symbol_id\"]]=n\n        for kind,rows in [(\"method_call\",graph[\"method_sites\"][\"sites\"]),(\"macro_invocation\",graph[\"graph\"][\"macro_invocations\"][\"rows\"])]:\n            raw_site_counts[(path,kind)]+=len(rows)\n            for site in rows:\n                assert site[\"source_oid\"]==paths[path][\"git_blob_oid\"]\n                assert site[\"site_id\"] not in sites\n                sites[site[\"site_id\"]]=(kind,site)\n    for col,nodes in [(\"published\",pub),(\"current\",current)]:\n        rows=syms[syms[col+\"_status\"]==\"represented\"].to_dict(\"records\")\n        ids=[r[col+\"_symbol_id\"] for r in rows];assert len(set(ids))==len(ids)\n        for r in rows:\n            n=nodes[r[col+\"_symbol_id\"]];assert n[\"file_path\"]==r[\"file_path\"]\n            end=n[\"byte_range_end\"] if col==\"published\" else n[\"byte_range\"][1]\n            assert end==r[\"end\"]\n            if col==\"published\":assert fm[r[\"file_path\"]]==paths[r[\"file_path\"]][\"git_blob_oid\"]\n        assert syms[col+\"_status\"].value_counts().to_dict()==summary[col+\"_declaration_states\"]\n    matched=calls[calls.current_status==\"represented\"].to_dict(\"records\")\n    assert len({r[\"current_site_id\"] for r in matched})==len(matched)\n    for r in matched:\n        kind,n=sites[r[\"current_site_id\"]];assert kind==r[\"kind\"] and n[\"file_path\"]==r[\"file_path\"]\n        assert n[\"byte_range\"]==[r[\"core_start\"],r[\"end\"]]\n        if kind==\"method_call\":assert n[\"method_byte_range\"]==[r[\"ident_start\"],r[\"ident_end\"]]\n    unmatched_site_ids=set(sites)-{r[\"current_site_id\"] for r in matched}\n    excluded={r[\"file_path\"] for r in parses if r[\"status\"]!=\"parsed\"}\n    assert all(sites[sid][1][\"file_path\"] in excluded for sid in unmatched_site_ids)\n    sg={p:g for p,g in syms.groupby(\"file_path\")};cg={p:g for p,g in calls.groupby(\"file_path\")}\n    for f in files.to_dict(\"records\"):\n        p=f[\"file_path\"];s=sg.get(p,syms.iloc[:0]);c=cg.get(p,calls.iloc[:0])\n        assert f[\"symbol_total\"]==len(s) and f[\"published_symbols\"]==int((s.published_status==\"represented\").sum())\n        assert f[\"current_symbols\"]==int((s.current_status==\"represented\").sum())\n        assert f[\"published_symbols\"]<=f[\"symbol_total\"] and f[\"current_symbols\"]<=f[\"symbol_total\"]\n        assert f[\"current_site_matches\"]==int((c.current_status==\"represented\").sum())\n        expected=\"unindexed\" if p not in fm else \"source_match\" if fm[p]==paths[p][\"git_blob_oid\"] else \"stale\"\n        assert f[\"published_file_status\"]==expected\n    assert files.published_file_status.value_counts().to_dict()==summary[\"published_file_states\"]\n    assert files.current_extraction_status.value_counts().to_dict()==summary[\"current_extraction_states\"]\n    assert inp.js(\"calibration-verification.json\")[\"status\"]==\"passed\"\n    unsupported=inp.js(\"unsupported-node-check.json\")\n    assert len(unsupported[\"published\"])==summary[\"published_declaration_states\"][\"no_matching_symbol\"]\n    assert len(unsupported[\"current\"])==summary[\"current_declaration_states\"][\"no_matching_symbol\"]\n    browser=inp.js(\"browser-verification.json\");assert browser[\"status\"]==\"passed\" and browser[\"initial_visible_files\"]==len(paths)\n    page=Page();page.feed(inp.read(\"coverage.html\").decode())\n    assert not page.external and page.file_rows==page.tile_count==len(paths)\n    page_data=json.loads(\"\".join(page.parts))\n    assert len(page_data[\"files\"])==len(files) and len(page_data[\"symbols\"])==len(syms)\n    if (Path(root)/\"payload-manifest.json\").exists():\n        for name,record in inp.js(\"payload-manifest.json\").items():\n            b=inp.read(name);assert len(b)==record[\"bytes\"] and hashlib.sha256(b).hexdigest()==record[\"sha256\"]\n    return {\"status\":\"passed\",\"source_files\":len(paths),\"ast_rows\":len(ast),\"declarations\":len(syms),\n        \"callsite_rows\":len(calls),\"current_successful_captures\":sum(r[\"status\"]==\"success\" for r in recs),\n        \"source_bound_site_matches\":len(matched),\"identical_external_nodes_shared_across_captures\":external_repeats,\"graph_only_sites_in_excluded_fragment\":len(unmatched_site_ids),\n        \"normalized_expression_attributes\":len(inp.js(\"method-gap-review.json\")),\n        \"html_file_rows\":page.file_rows,\"html_tiles\":page.tile_count,\"html_external_resources\":len(page.external),\n        \"browser\":browser[\"status\"],\"uses_private_solver_store\":False}\n\nif __name__==\"__main__\":\n    import sys\n    result=verify(Path(sys.argv[1]) if len(sys.argv)>1 else Path(__file__).parent)\n    print(json.dumps(result,indent=2))\n")
verify_spec=importlib.util.spec_from_file_location('rust_coverage_verify',OUT/'verify.py')
coverage_verify=importlib.util.module_from_spec(verify_spec)
verify_spec.loader.exec_module(coverage_verify)
verification=coverage_verify.verify(OUT)
dump('verification.json',verification)
print(json.dumps(verification,indent=2))


{
  "status": "passed",
  "source_files": 689,
  "ast_rows": 205726,
  "declarations": 26507,
  "callsite_rows": 165689,
  "current_successful_captures": 683,
  "source_bound_site_matches": 96903,
  "identical_external_nodes_shared_across_captures": 2117,
  "graph_only_sites_in_excluded_fragment": 126,
  "normalized_expression_attributes": 8,
  "html_file_rows": 689,
  "html_tiles": 689,
  "html_external_resources": 0,
  "browser": "passed",
  "uses_private_solver_store": false
}


In [34]:
seen={};duplicates=[]
for rec in extraction_records:
    if rec["status"]!="success":continue
    o=json.loads(gzip.decompress((OUT/"current-extractor"/rec["artifact"]).read_bytes()))
    for n in o["graph"]["symbols"]:
        sid=n["stable_symbol_id"]
        if sid in seen:
            duplicates.append({"first":seen[sid],"second":n,"capture":rec["path"]})
        else:seen[sid]=n
print("Raw current node ID repeats",len(duplicates),"sample",duplicates[:2])
dump("raw-node-identity-repeats.json",duplicates)


Raw current node ID repeats 2117 sample [{'first': {'stable_symbol_id': '030ce6dd4fe159b8', 'file_path': 'external://', 'byte_range': [0, 0], 'line_range': [0, 0], 'entity_name': 'Arc', 'qualified_name': 'std::sync::Arc', 'symbol_kind': 'external', 'anchor_hash': '', 'enclosing_scope': None}, 'second': {'stable_symbol_id': '030ce6dd4fe159b8', 'file_path': 'external://', 'byte_range': [0, 0], 'line_range': [0, 0], 'entity_name': 'Arc', 'qualified_name': 'std::sync::Arc', 'symbol_kind': 'external', 'anchor_hash': '', 'enclosing_scope': None}, 'capture': 'app_gallery/spur-linear/backend/rust/metrics.rs'}, {'first': {'stable_symbol_id': 'c19c041d63aa7711', 'file_path': 'external://', 'byte_range': [0, 0], 'line_range': [0, 0], 'entity_name': 'RecordBatch', 'qualified_name': 'arrow::record_batch::RecordBatch', 'symbol_kind': 'external', 'anchor_hash': '', 'enclosing_scope': None}, 'second': {'stable_symbol_id': 'c19c041d63aa7711', 'file_path': 'external://', 'byte_range': [0, 0], 'line_rang

In [36]:
import importlib.metadata, tarfile, io
state_after={"head":git("rev-parse","HEAD").decode().strip(),"status":git("status","--porcelain=v1","-z").decode(),"graph_current":str((ROOT/".spur/graph/CURRENT").resolve()),"graph_manifest_sha256":sha((CURRENT/"manifest.json").read_bytes())}
live_mismatches=[r["path"] for r in source_manifest if not (ROOT/r["path"]).is_file() or sha((ROOT/r["path"]).read_bytes())!=r["sha256"]]
dump("project-state-after.json",state_after)
protection={"source_files_checked":len(source_manifest),"source_mismatches":live_mismatches,"same_head":state_before["head"]==state_after["head"],"same_git_status":state_before["status"]==state_after["status"],"same_graph_pointer":state_before["graph_current"]==state_after["graph_current"],"same_graph_manifest":state_before["graph_manifest_sha256"]==state_after["graph_manifest_sha256"],"target_writes_by_experiment":False}
dump("target-protection.json",protection)
assert not live_mismatches and all(protection[k] for k in ["same_head","same_git_status","same_graph_pointer","same_graph_manifest"]),protection
versions={n:importlib.metadata.version(n) for n in ["pandas","pyarrow","duckdb","matplotlib","tornado"]}
versions.update(python=sys.version,syn="2.0.118",proc_macro2="1, span-locations feature",rust_execution="native notebook Rust cell")
dump("environment.json",versions)
readme=f"""# Rust source and Spur graph coverage

Open [the interactive report](coverage.html), [the overview](coverage-overview.png), or the executed notebook at §../../../../2026-11-02-rust-graph-coverage.ipynb§. The report includes a clickable 689-file map, file/category filters, declaration line spans and per-symbol drilldowns. The notebook adds GGSQL and Perspective views.

Root task: **bd-33a**. Independent audit: **bd-l8h**, source lineage **bd-1xw**, approved plan §5d856f61-02d0-4341-811b-0e6cae45ce02§. Production code in both projects was left unchanged.

## Measured result

| Layer | Result |
|---|---|
| Physical Rust source | 689 tracked/nonignored Rust files; 366,382 physical lines; 13,351,580 bytes |
| Independent parser | Native Rust notebook, syn 2.0.118: 688 complete files parsed; one directive-bearing fragment excluded |
| Published file freshness | **651 source matches, 17 stale, 21 unindexed** |
| Published callable declarations | **11,517 / 12,682 verified (90.8%)**; 797 stale, 368 unindexed |
| Callables within source-matching files | **11,517 / 11,517 represented** |
| All declaration kinds | 23,197 / 26,507 represented; 2,144 stale; 790 unindexed; 376 without individual symbol correspondence |
| Current file-local extractor | **683 successful captures, six 30-second timeouts** |
| Current callables in successful files | **11,291 / 11,291 represented**; 1,391 further callable declarations unmeasured |
| Current method identities | **72,149 exact matches**; 9,797 further AST sites unmeasured |
| Current macro identities | **24,754 exact matches**; 3,608 further AST sites unmeasured |
| Free-call invocation identities | Not exposed by this capture API; no recall percentage claimed |

These are different eligible populations, not a before/after improvement estimate. Coverage is **physical source representation**, not executed line coverage or proof that code cannot panic. Callables include ordinary functions, methods and trait methods; nine foreign-function declarations are separate.

## Source and graph binding

Target: §../spur-notebook§, captured HEAD §{state_before["head"]}§. Every file has a SHA-256 and Git blob OID in §source-manifest.json§. The denominator comes from §git ls-files --cached --others --exclude-standard -- '*.rs'§, restricted to existing files. Ignored build outputs and external dependency sources are excluded.

The published graph is **spur-graph-schema-v10**, hash §{GRAPH_MANIFEST["graph_content_hash"]}§. Its §complete=true§ describes artifact construction, not coverage of the current worktree. The registry points at unavailable §/Volumes/Projects/spur-notebook§; schema-discovered, read-only Parquet access was used. CURRENT was not rebuilt or replaced.

Current extraction used the previously tested binary, source commit §848a29f39511cbd27091beb6c171b97e179f353a§, SHA-256 §{BIN_SHA}§. Each file was extracted in an isolated private root. This measures syntax collection, with **no project-wide target-resolution claim**. Captured JSON header version is not a persisted-v11 completeness certificate; its method/macro rows are separate from published v10.

Successful captures retain compressed original stdout, request, duration, source identity and stdout/binary hashes. Six files exceeded the explicit 30-second budget at concurrency six. That is an experimental limit, not unsupported syntax. Their paths are in §summary.json§.

## What is counted

The independent syn visitor enumerates declarations, fields, variants and implementation blocks from physical source. Calls, macros, closures, async/const blocks and index operations are separate grains. Macro token trees remain opaque; no generated-item denominator is invented.

Symbol matching requires equal source identity, file, compatible kind, normalized raw-identifier name, identifier containment and exact declaration end. Implementation containers use file/kind/end containment. Represented symbol/site IDs are used at most once. Stale files are rejected before matching.

Method matching uses source identity, name, receiver start and exact call end. Eight expression attributes initially changed the full AST span; exact method-token positions independently verify their normalization. Nested/repeated sites remain distinct. See §calibration-verification.json§ and §method-gap-review.json§.

The excluded §jute-notebook/src-tauri/src/assets/ports_bootstrap.rs§ starts with notebook dependency directives. Its 115 method and 11 macro rows are excluded from numerator and denominator alike. Dividing raw capture counts by AST counts would exceed 100%.

Categories separate production/mixed paths, test files, examples, build tools, vendor and archived/research copies. Unit tests can remain in mixed files. The §cfg_context§ / §test_context§ fields are inherited **item annotation hints**, not exhaustive expression metadata or evaluated cfg formulas. False flags do not imply unconditional or production-only code. These flags do not drive top-line coverage. No captured file had an explicit generated-comment marker in its first 20 lines; no marker does not prove handwritten origin.

## Remaining gaps

- Published v10: 38 files need freshness/index coverage before current declarations can be trusted.
- Observed tuple fields and foreign declarations lack individual correspondence. Direct node inspection checked all **376 published** and **406 current** gaps (§unsupported-node-check.json§). Parent type text may still contain those bytes.
- Free-call identity remains unmeasured by this API. The inventory additionally records **8,808 body forms** and **4,722 indexing operations**, without inferring execution or panic.
- Six timeout files contain 3,031 declarations and remain unmeasured.
- Cargo configurations, macro expansions, trait dispatch, implicit compiler checks, value flow, branch feasibility and runtime reachability need separate evidence. No complete-panic-detection percentage is inferred.

Practical next priorities are refreshing the target's published graph with a compatible reader, exposing/persisting general invocation/body evidence, and separately investigating measured extraction timeouts. Those implementation changes were not made here.

## Verification and replay

§verification.json§ verifies source bytes/OIDs, AST ranges/identifiers, capture hashes, source bindings, per-file bounds, unique represented identities, method tokens, parser exclusions and saved HTML. Identical external endpoint nodes repeated across independent captures are permitted; physical represented declaration IDs remain unique.

The offline browser check verified 689 initial rows, 21 unindexed rows, 175 test-file rows, empty search, keyboard selection, declaration drilldown and no horizontal overflow. Charts and all file rows are pre-rendered; HTML uses no external resources.

§audit/REPORT.md§ records the independent audit with hashes and counterexamples. §target-protection.json§ verifies unchanged target HEAD, Git status, source bytes, graph pointer and manifest. No solver store was read and no project build was needed.

Standalone verification (Python with pandas/pyarrow):
§§§sh
python verify.py .
§§§

This reads §evidence.tar.gz§ when raw loose inputs are absent; it does not extract or parse notebooks. To regenerate tables in a disposable copy of this artifact directory:

§§§sh
python -m tarfile -e evidence.tar.gz .
python analyze.py .
python verify.py .
§§§

The notebook is the executable source for Rust inventory, capture commands, SQL views, Arrow port and HTML rendering. §inventory.evcxr.rs§ and §*.cell.*§ preserve exact cell sources; they require notebook kernel state and are not standalone entrypoints. Dependencies/versions are in §inventory-dependencies.txt§ and §environment.json§. Re-running live source capture creates a new measurement; do not mix new inputs with old captures.

Raw sources, AST rows, published graph tables and original captures are bundled, with member hashes in §payload-manifest.json§. The integration commit uses a dedicated plan branch; the user's active checkout was preserved.
""".replace("§",chr(96))
(OUT/"README.md").write_text(readme)
raw_paths=sorted([p for p in (OUT/"source-snapshot").rglob("*") if p.is_file()]+list((OUT/"current-extractor").glob("*.json.gz"))+
    [OUT/"rust-ast.json",OUT/"graph-nodes.parquet",OUT/"graph-file_manifests.parquet",OUT/"graph-edges.parquet",OUT/"graph-edges_unresolved.parquet",OUT/"raw-node-identity-repeats.json"])
payload_manifest={p.relative_to(OUT).as_posix():{"bytes":p.stat().st_size,"sha256":sha(p.read_bytes())} for p in raw_paths}
dump("payload-manifest.json",payload_manifest)
with open(OUT/"evidence.tar.gz","wb") as dst:
    with gzip.GzipFile(fileobj=dst,mode="wb",mtime=0,compresslevel=6) as gz:
        with tarfile.open(fileobj=gz,mode="w") as tar:
            for p in raw_paths:
                content=p.read_bytes();info=tarfile.TarInfo(p.relative_to(OUT).as_posix())
                info.size=len(content);info.mode=0o644;info.mtime=0;info.uid=info.gid=0;info.uname=info.gname=""
                tar.addfile(info,io.BytesIO(content))
dump("bundle-summary.json",{"members":len(raw_paths),"uncompressed_bytes":sum(r["bytes"] for r in payload_manifest.values()),"archive_bytes":(OUT/"evidence.tar.gz").stat().st_size,"archive_sha256":sha((OUT/"evidence.tar.gz").read_bytes())})
print("Target protection",protection)
print("Archive",json.loads((OUT/"bundle-summary.json").read_text()))
print("README and reproducible input bundle prepared.")


Target protection {'source_files_checked': 689, 'source_mismatches': [], 'same_head': True, 'same_git_status': True, 'same_graph_pointer': True, 'same_graph_manifest': True, 'target_writes_by_experiment': False}
Archive {'members': 1379, 'uncompressed_bytes': 90492994, 'archive_bytes': 24726800, 'archive_sha256': 'b5d532113e7e7e502f2672d9b056012f536181a12cb9f29da204c67d187fb385'}
README and reproducible input bundle prepared.


In [47]:
CURATED=[
"README.md","analyze.py","verify.py","inventory.evcxr.rs","inventory-dependencies.txt","render-report.cell.py","render-charts.cell.py","browser-check.cell.txt",
"coverage.html","dashboard.js","coverage-overview.svg","coverage-overview.png","coverage-by-kind.svg","coverage-by-kind.png","dashboard-screenshot.png",
"source-manifest.json","graph-manifest.json","rust-parse-results.json","calibration.rs","calibration-ast.json","calibration-current-extractor.json","calibration-verification.json","method-gap-review.json",
"current-extractor-provenance.json","current-extractor-results.json","summary.json","file-coverage.parquet","symbol-coverage.parquet","callsite-coverage.parquet","published-kind-coverage.parquet","current-kind-coverage.parquet",
"file-coverage.csv","published-symbol-gaps.csv","current-symbol-gaps.csv","unsupported-node-check.json","browser-verification.json","verification.json","environment.json",
"project-state-before.json","project-state-after.json","target-protection.json","worker-tracker-audits.json","payload-manifest.json","bundle-summary.json","evidence.tar.gz","audit/REPORT.md"
]
DELIVERY=Path("/private/tmp/r8g-root-review-worktree/docs/research/artifacts/2026-11-02-rust-graph-coverage")
DELIVERY.mkdir(parents=True,exist_ok=True)
for name in CURATED:
    dest=DELIVERY/name;dest.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(OUT/name,dest)
assert not (DELIVERY/"source-snapshot").exists()
check=subprocess.run([sys.executable,str(DELIVERY/"verify.py"),str(DELIVERY)],capture_output=True,text=True,timeout=60)
assert check.returncode==0,check.stderr[-2000:]
archive_verification=json.loads(check.stdout)
assert archive_verification==verification,(archive_verification,verification)
dump("archive-verification.json",{"exit":check.returncode,"result":archive_verification,"stderr":check.stderr,"mode":"curated-only delivery, raw inputs loaded from evidence archive"})
shutil.copyfile(OUT/"archive-verification.json",DELIVERY/"archive-verification.json")
# Root's archive fallback has now been proved without relying on loose capture inputs.
# Remove only this experiment's raw duplicates; the frozen bytes remain in the verified archive.
for name in ["source-snapshot","current-extractor"]:
    if (OUT/name).exists():shutil.rmtree(OUT/name)
for name in ["rust-ast.json","graph-nodes.parquet","graph-file_manifests.parquet","graph-edges.parquet","graph-edges_unresolved.parquet","raw-node-identity-repeats.json","symbol-coverage.json"]:
    if (OUT/name).exists():(OUT/name).unlink()
root_archive_result=coverage_verify.verify(OUT)
assert root_archive_result==archive_verification
dump("curated-files.json",CURATED+["archive-verification.json"])
shutil.copyfile(OUT/"curated-files.json",DELIVERY/"curated-files.json")
print("Archive-only verification passed in both the isolated delivery directory and original report directory.")
print("Curated files",len(CURATED)+2,"total bytes",sum((DELIVERY/name).stat().st_size for name in CURATED))
print("Raw source/capture duplicates removed after archive verification. All chart/table inputs remain available.")


Archive-only verification passed in both the isolated delivery directory and original report directory.
Curated files 48 total bytes 45945192
Raw source/capture duplicates removed after archive verification. All chart/table inputs remain available.
